# Cognito — Sistema de Inferência com Paging Dinâmico de KV Cache em Nível de Aplicação

**Trabalho de Conclusão de Curso — Anexo Técnico de Implementação**

Este notebook documenta a implementação *State-of-the-Art* do sistema Cognito. O pipeline soluciona ativamente o paradoxo de RAG de Longo Contexto vs Restrição de Hardware na Borda, operando através de um mecanismo inovador de **Poda Semântica de Memória e Paging Dinâmico de Contextos**.

Organização do Pipeline:

| Fase | Script | Descrição |
|------|--------|-----------|
| 1 | `1_ingestao.py` | Ingestão com **Semantic Chunking** e indexação vetorial densa. |
| 2 | `2_avaliacao.py` | Benchmarking padronizado via LM Evaluation Harness (Validando integridade NF4). |
| 3 | `3_inferencia.py` | Engine RAG com **RoPE Tracking**, decaimento de **Ebbinghaus (Multi-Turno)** e evicção via Entropia/Cross-Encoder. |

**Ambiente Experimental:** GPU NVIDIA T4 Limitada em ~5.4GB (Restrição de 35% VRAM simulando IA de Borda)  
**Modelo Alvo:** `mistralai/Mistral-7B-Instruct-v0.3` (NF4 Quantized)  
**Bibliotecas Core:** PyTorch (SDPA), Transformers, ChromaDB, Sentence-Transformers  

---

**Rigor e Metodologia Científica Acoplada:**  
O código executa processos blindados contra vazamentos (Isolamento de Kernel via `uv run`), e garante estabilidade absoluta da ordem de geração (**Rotary Position Embedding Estabilizado**) mesmo quando partes imensas do contexto são apagadas da placa de vídeo para dar espaço à nova geração. O sistema mede sucesso pelo **Contains Accuracy**, avaliando latências exatas (TTFT e ITL).


## 0. Gestão de Sessão — Google Drive

Execute a **Célula de Início** ao abrir o Colab e a **Célula de Fim** antes de fechar.

| O que persiste no Drive | Por quê |
|-------------------------|----------|
| `chroma_cognito/` | ChromaDB — caro de reconstruir (~10 min embedding) |
| `gold_answers.json` | Queries de avaliação e gabaritos |
| `results_checkpoint_C*.jsonl` | Resultados — perder esses é perder sessões de GPU |
| Modelo Mistral | **Não persiste** — HF Hub re-download (~1 min) é mais rápido que Drive |

> **Estratégia de I/O:** ChromaDB usa SQLite com leituras aleatórias intensas.  
> Rodar direto do Drive seria 3-5× mais lento. A cópia Drive → `/content/` leva ~15-30s e elimina o gargalo.


In [1]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE INÍCIO DE SESSÃO — Execute SEMPRE ao abrir o Colab
# ═══════════════════════════════════════════════════════════════
# Monta o Drive e copia dados persistentes para o SSD local (/content/).
# ChromaDB tem I/O aleatório intenso — rodar direto do Drive seria
# 3-5x mais lento. A cópia local demora ~10-30s e resolve o problema.

import os, shutil, time
from google.colab import drive

drive.mount('/content/drive')

# ── Configuração de caminhos ─────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT  = '/content'

# Cria diretórios no Drive se não existirem
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/chroma_cognito', exist_ok=True)

# ── Restaura ChromaDB (índice vetorial) ──────────────────────────
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
chroma_sqlite = f'{DRIVE_CHROMA}/chroma.sqlite3'

if os.path.exists(chroma_sqlite):
    print('Restaurando ChromaDB do Drive → /content/ ...')
    t0 = time.time()
    if os.path.exists(LOCAL_CHROMA):
        shutil.rmtree(LOCAL_CHROMA)
    shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB restaurado: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
else:
    print('⚠ ChromaDB não encontrado no Drive.')
    print('  Execute a célula de ingestão (1_ingestao.py) para construí-lo.')

# ── Restaura gold_answers.json ────────────────────────────────────
DRIVE_GOLD = f'{DRIVE_ROOT}/gold_answers.json'
if os.path.exists(DRIVE_GOLD):
    shutil.copy(DRIVE_GOLD, f'{LOCAL_ROOT}/gold_answers.json')
    import json as _j
    n = len(_j.load(open(f'{LOCAL_ROOT}/gold_answers.json')))
    print(f'  ✓ gold_answers.json restaurado: {n} queries')
else:
    print('  ⚠ gold_answers.json não encontrado — será gerado pela ingestão.')

# ── Restaura checkpoints de runs anteriores ───────────────────────
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'
ckpts = [f for f in os.listdir(DRIVE_CKPTS) if f.endswith('.jsonl')]
for ck in ckpts:
    shutil.copy(f'{DRIVE_CKPTS}/{ck}', f'{LOCAL_ROOT}/{ck}')
if ckpts:
    print(f'  ✓ Checkpoints restaurados: {ckpts}')
else:
    print('  ℹ Nenhum checkpoint anterior encontrado (primeira sessão).')

print()
print('✓ Sessão restaurada. Dados em /content/ prontos para uso.')
print(f'  Drive root: {DRIVE_ROOT}')


Mounted at /content/drive
⚠ ChromaDB não encontrado no Drive.
  Execute a célula de ingestão (1_ingestao.py) para construí-lo.
  ⚠ gold_answers.json não encontrado — será gerado pela ingestão.
  ℹ Nenhum checkpoint anterior encontrado (primeira sessão).

✓ Sessão restaurada. Dados em /content/ prontos para uso.
  Drive root: /content/drive/MyDrive/cognito_tcc


## 1. Instalação de Dependências

In [2]:
%%writefile 1_ingestao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "datasets>=2.20.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "torch",
#     "einops"
# ]
# ///

import json
import re
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
VECTOR_DB_PATH       = "./chroma_cognito"

# ── Separação Corpus / Avaliação (Fase 0 — Roadmap) ──────────────────────────
# Documentos indexados vêm do split TRAIN do TriviaQA.
# Queries de avaliação vêm do split VALIDATION (independente).
# [Fase 0] Expansão do corpus para subir gold_in_corpus (alvo ≥ 60 %).
# Rodada anterior: train[:600], MAX=1000 → gold_in_corpus=41.5 %, abaixo do
# paramétrico (C1 EM=48 %). Petroni 2021 (KILT) recomenda fixar o teto de
# retrievability antes de avaliar RAG. Custo: ~15 min de ingestão a mais.
CORPUS_SPLIT     = "train[:3000]"
EVAL_SPLIT       = "validation[:200]"

MIN_PASSAGE_CHARS   = 500
MAX_CORPUS_PASSAGES = 5000


def main():
    # ── Corpus (train) ───────────────────────────────────────────────────────
    print("Carregando corpus TriviaQA (split TRAIN para indexação)...")
    train_data = load_dataset("trivia_qa", "rc.wikipedia", split=CORPUS_SPLIT)

    CHUNK_CHARS   = 1024
    OVERLAP_CHARS = 256

    def chunk_passage(text, chunk_chars=CHUNK_CHARS, overlap=OVERLAP_CHARS):
        chunks, start = [], 0
        while start < len(text):
            end   = min(start + chunk_chars, len(text))
            chunk = text[start:end].strip()
            if len(chunk) >= 150:
                chunks.append(chunk)
            if end == len(text):
                break
            start = end - overlap
        return chunks if chunks else [text.strip()]

    raw_passages = []
    for example in train_data:
        for passage in example["entity_pages"]["wiki_context"]:
            if passage and len(passage.strip()) >= MIN_PASSAGE_CHARS:
                raw_passages.append(passage.strip())

    seen, dedup = set(), []
    for doc in raw_passages:
        key = doc[:512]
        if key not in seen:
            seen.add(key)
            dedup.append(doc)

    documents = []
    for doc in dedup:
        documents.extend(chunk_passage(doc))
    documents = documents[:MAX_CORPUS_PASSAGES]

    print(f"Corpus: {len(documents)} chunks indexados (de {len(dedup)} passagens unicas).")
    print(f"Comprimento medio: {sum(len(d) for d in documents) // max(len(documents),1)} chars.")

    # ── Gold Answers (validation) ─────────────────────────────────────────────
    print("Carregando queries de avaliacao (split VALIDATION)...")
    val_data = load_dataset("trivia_qa", "rc.wikipedia", split=EVAL_SPLIT)

    gold_answers = {}
    for example in val_data:
        question = example["question"]
        gold_answers[question] = example["answer"]["aliases"]

    print(f"Queries de avaliacao disponiveis: {len(gold_answers)}.")

    with open("gold_answers.json", "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, ensure_ascii=False)

    # ── [P1.3] Gold-presence validation (KILT methodology, Petroni 2021) ─────
    # Para isolar falha de retrieval de falha de geracao, validar se a
    # resposta-gold esta presente no corpus indexado. Substring case-
    # insensitive sobre todas as aliases. Define o teto de recall (retrievability
    # ceiling) para o subset filtrado da avaliacao.
    print("Computando gold-presence no corpus indexado (KILT)...")
    docs_lower = [d.lower() for d in documents]

    def alias_in_corpus(aliases):
        for a in aliases:
            a_norm = re.sub(r"\s+", " ", a.strip().lower())
            if len(a_norm) < 2:
                continue
            for d in docs_lower:
                if a_norm in d:
                    return True
        return False

    gold_hits = {}
    for q, aliases in gold_answers.items():
        gold_hits[q] = alias_in_corpus(aliases)

    n_hits = sum(1 for v in gold_hits.values() if v)
    pct = n_hits / max(1, len(gold_hits)) * 100
    print(f"Gold-presence: {n_hits}/{len(gold_hits)} questoes ({pct:.1f}%).")
    print(f"  Retrievability ceiling: EM maximo possivel com retrieval ideal = {pct:.1f}%.")

    with open("corpus_gold_hits.json", "w", encoding="utf-8") as f:
        json.dump(gold_hits, f, ensure_ascii=False)

    # ── Indexacao ─────────────────────────────────────────────────────────────
    print("Calculando embeddings e escrevendo no DB local...")
    client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
    try:
        client.delete_collection("cognito_knowledge_base")
    except Exception:
        pass

    collection = client.create_collection(
        name="cognito_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
    embeddings  = embed_model.encode(
        documents,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))],
    )
    print("Fase 1 concluida com sucesso.")
    print(f"  Corpus (TRAIN):      {len(documents)} passagens indexadas.")
    print(f"  Avaliacao (VAL):     {len(gold_answers)} queries com gabarito.")
    print(f"  Gold-presence:       {n_hits}/{len(gold_hits)} no corpus.")
    print("  Data leakage: ZERO — splits sao disjuntos por design.")


if __name__ == "__main__":
    main()


Writing 1_ingestao.py


In [3]:
!pip install -q uv
!uv run 1_ingestao.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 23.8 MB/s eta 0:00:00
Installed 122 packages in 1.25s
Carregando corpus TriviaQA (split TRAIN para indexação)...
README.md: 26.7kB [00:00, 47.8MB/s]
Resolving data files: 100% 26/26 [00:00<00:00, 22866.83it/s]
rc.wikipedia/train-00000-of-00007.parque(…): 100% 240M/240M [00:03<00:00, 64.4MB/s]
rc.wikipedia/train-00001-of-00007.parque(…): 100% 261M/261M [00:07<00:00, 36.2MB/s]
rc.wikipedia/train-00002-of-00007.parque(…): 100% 319M/319M [00:07<00:00, 45.4MB/s]
rc.wikipedia/train-00003-of-00007.parque(…): 100% 266M/266M [00:04<00:00, 57.6MB/s]
rc.wikipedia/train-00004-of-00007.parque(…): 100% 240M/240M [00:05<00:00, 47.9MB/s]
rc.wikipedia/train-00005-of-00007.parque(…): 100% 259M/259M [00:08<00:00, 32.3MB/s]
rc.wikipedia/train-00006-of-00007.parque(…): 100% 253M/253M [00:04<00:00, 52.5MB/s]
rc.wikipedia/validation-00000-of-00001.p(…): 100% 235M/235M [00:07<00:00, 32.6MB/s]
rc.wikipedia/test-00000-of-00001.parquet: 100% 221M/221M [00:

## Fase 2: Baseline do Modelo — Mistral-7B-Instruct-v0.3

Resultados reportados por referência cruzada: HuggingFace Open LLM Leaderboard e Jiang et al. (2023).
Quantização NF4 introduz degradação < 2 pp vs. float16 (Dettmers et al., 2023, QLoRA).
Execução completa via `lm-eval` requer ~40–60 min em A100; inviável na T4 gratuita do Colab.

| Benchmark | Métrica | Score | Fonte |
|-----------|---------|-------|-------|
| ARC Challenge | acc\_norm | **59.98%** | HF Open LLM Leaderboard |
| HellaSwag | acc\_norm | **81.30%** | HF Open LLM Leaderboard |
| MMLU | acc | **60.10%** | Jiang et al. (2023) |
| Winogrande | acc | **75.30%** | HF Open LLM Leaderboard |

> **Nota:** Estes resultados caracterizam a capacidade paramétrica base do modelo.
> O objetivo do Cognito não é superar este baseline em benchmarks gerais,
> mas preservar qualidade em inferência long-context com menor VRAM.

## Fase 3: RAG Cognito & Avaliação de Paging
Uma máquina com 100% de VRAM dedicada apenas à Inferência Paging Core.


In [12]:
%%writefile 3_inferencia.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "numpy>=1.26.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "peft>=0.12.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "scikit-learn>=1.5.1",
#     "torch",
#     "einops",
#     "rank_bm25>=0.2.2",
#     "hqq>=0.2.0",
#     "datasets>=2.20.0",
# ]
# ///

import gc
import json
import math
import os
import random
import re
import string
import sys
import time
from collections import Counter

import numpy as np
import torch
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
)
import warnings
warnings.filterwarnings("ignore")

# ─── Constants ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
RERANKER_MODEL_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME       = "mistralai/Mistral-7B-Instruct-v0.3"
VECTOR_DB_PATH       = "./chroma_cognito"

MAX_CONTEXT_CHARS                 = 40_000
MAX_NEW_TOKENS                    = 64
NUM_QUERIES_EVAL                  = 50

PAGING_CONTEXT_THRESHOLD_TOKENS   = 1000
CHUNKED_PREFILL_CHUNK_SIZE        = 256       # cf. Sarathi-Serve, OSDI 2024
CHUNKED_PREFILL_THRESHOLD_TOKENS  = 2048

PROMPT_LOOKUP_NUM_TOKENS = 10                 # input-grounded n-gram speculation
USE_HQQ_KV_CACHE         = False
FEWSHOT_K                = 2
HYBRID_RRF_K             = 60                 # Reciprocal-Rank-Fusion (Cormack 2009)
BOOTSTRAP_RESAMPLES      = 1000

# ── [P1.2] CRAG: thresholds absolutos no logit raw do cross-encoder ──────────
# ms-marco-MiniLM-L-6-v2 logits aproximadamente em [-10, 10]; pares relevantes
# no TriviaQA tipicamente > 2.0. Yan et al. 2024 ("Corrective RAG"): tres
# buckets — trust / ambiguous / no_context.
CRAG_TRUST_THRESH       = 2.0
CRAG_NOCONTEXT_THRESH   = -3.0

# ── [P1.1] Gate adaptativo de retrieval ──────────────────────────────────────
# Mallen et al. ACL 2023 ("When Not to Trust Language Models"): RAG fere o
# baseline parametrico em factoids comuns. Asai et al. ICLR 2024 ("Self-RAG"):
# decisao de recuperar deve ser adaptativa. Threshold no top-1 raw cross-
# encoder score: abaixo do threshold cai para parametrico.
#
# [Fase 2] Threshold a ser calibrado empiricamente em main() via
# `calibrate_adaptive_gate(...)` — alvo: disparar RAG em 30-60% das queries
# (rodada anterior com 1.0 disparou 1/50). Default fica conservador caso a
# calibração falhe.
ADAPTIVE_RETRIEVAL_GATE      = 1.0
ADAPTIVE_GATE_TARGET_FIRE    = 0.50   # fração de queries que devem acionar RAG
ADAPTIVE_GATE_FIRE_BOUNDS    = (0.20, 0.80)  # [Fase 2] critério de aceitação

# ─── [P0] Eval verbosity control ─────────────────────────────────────────
# Per-query prints inflate stdout to >50k chars for 50q × 8cfg runs and
# break downstream context windows. Default OFF in production; ON only
# under --test (small sample, debugging). Detalhe completo sempre vai
# para eval_traces.jsonl para post-mortem.
EVAL_VERBOSE                 = False
EVAL_TRACE_PATH              = "eval_traces.jsonl"
# ─── [P2] Eval logging ────────────────────────────────────────────────
EVAL_LOG_PATH = "eval_results.txt"

def append_log(msg: str):
    """Append msg to eval_results.txt with newline."""
    with open(EVAL_LOG_PATH, "a") as f:
        f.write(msg + "\n")

EVAL_PROGRESS_EVERY          = 10  # print 1 progress line every N queries quando quiet


# ─── Generic utilities ────────────────────────────────────────────────────
def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def _get_kv_tensors(cache):
    """Extract k/v lists from any DynamicCache layout (legacy / 5.x / tuple)."""
    if hasattr(cache, "key_cache") and isinstance(getattr(cache, "key_cache"), list):
        kc = cache.key_cache
        if len(kc) > 0 and hasattr(kc[0], "shape"):
            return kc, cache.value_cache
    if hasattr(cache, "layers") and len(cache.layers) > 0:
        layer0 = cache.layers[0]
        if hasattr(layer0, "keys") and layer0.keys is not None and layer0.keys.numel() > 0:
            keys = [l.keys for l in cache.layers if hasattr(l, "keys") and l.keys is not None]
            vals = [l.values for l in cache.layers if hasattr(l, "values") and l.values is not None]
            if keys:
                return keys, vals
    try:
        return [layer[0] for layer in cache], [layer[1] for layer in cache]
    except (TypeError, IndexError):
        raise ValueError(f"Cannot extract KV tensors from cache type: {type(cache)}")

def get_cache_seq_len(cache):
    if cache is None:
        return 0
    if hasattr(cache, "get_seq_length"):
        try:
            v = cache.get_seq_length()
            return int(v) if not isinstance(v, int) else v
        except Exception:
            pass
    try:
        keys, _ = _get_kv_tensors(cache)
        if keys:
            return keys[0].shape[-2]
    except Exception:
        pass
    return 0

def _rebuild_cache_from_tensors(key_list, value_list):
    from transformers.cache_utils import DynamicCache
    new_cache = DynamicCache()
    for layer_idx in range(len(key_list)):
        new_cache.update(key_list[layer_idx], value_list[layer_idx], layer_idx)
    return new_cache

def _normalize_text(s: str) -> str:
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', s.lower()).split()).translate(
        str.maketrans('', '', string.punctuation))


class _NullCtx:
    def __enter__(self): return None
    def __exit__(self, *a): return False


# ─── Pager classes ────────────────────────────────────────────────────────
class VirtualPageManager:
    """StreamingLLM-style offload (Xiao 2024, ICLR): keep `sink_size` initial
    tokens + recent window; middle slice asynchronously copied to pinned CPU
    memory so it overlaps with compute under a non-default CUDA stream."""
    def __init__(self, threshold_gb: float = 7.5, page_size: int = 1024, sink_size: int = 4):
        self.threshold_bytes  = threshold_gb * (1024 ** 3)
        self.page_size        = page_size
        self.sink_size        = sink_size
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False
        self._streamllm_offloads = 0

    def check_vram_pressure(self) -> bool:
        if not torch.cuda.is_available():
            return False
        return torch.cuda.memory_allocated(0) > self.threshold_bytes

    @staticmethod
    def _to_pinned_cpu(t):
        if t is None:
            return t
        cpu = torch.empty(t.shape, dtype=t.dtype, device='cpu', pin_memory=True)
        cpu.copy_(t.detach(), non_blocking=True)
        return cpu

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.check_vram_pressure():
            return past_key_values

        key_cache, value_cache = _get_kv_tensors(past_key_values)
        if not any(k.shape[-2] > self.page_size for k in key_cache):
            return past_key_values

        new_keys, new_values = [], []
        layers_paged = 0
        for layer_idx, (k, v) in enumerate(zip(key_cache, value_cache)):
            seq_len = k.shape[-2]
            if seq_len > self.page_size:
                keep = self.page_size - self.sink_size
                k_cpu = self._to_pinned_cpu(k[:, :, self.sink_size:-keep, :])
                v_cpu = self._to_pinned_cpu(v[:, :, self.sink_size:-keep, :])
                self.cpu_swap_storage.append((layer_idx, k_cpu, v_cpu))
                new_keys.append(torch.cat([k[:, :, :self.sink_size, :], k[:, :, -keep:, :]], dim=2))
                new_values.append(torch.cat([v[:, :, :self.sink_size, :], v[:, :, -keep:, :]], dim=2))
                layers_paged += 1
            else:
                new_keys.append(k); new_values.append(v)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        gc.collect(); torch.cuda.empty_cache()

        new_cache = _rebuild_cache_from_tensors(new_keys, new_values)
        if not self._debug_printed:
            print(f"[Pager] StreamingLLM offload | layers={layers_paged} | "
                  f"seq_len={get_cache_seq_len(new_cache)}")
            self._debug_printed = True
        if layers_paged > 0:
            self._streamllm_offloads += 1
        return new_cache

    def reset(self):
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False
        self._streamllm_offloads = 0
        force_cleanup()

    @property
    def blocks_in_cpu(self) -> int:
        return len(self.cpu_swap_storage)

    @property
    def streamllm_offloads(self) -> int:
        return self._streamllm_offloads


class AdaptiveVirtualPageManager(VirtualPageManager):
    def __init__(self, initial_threshold_gb: float = 7.5, page_size: int = 1024,
                 safety_margin: float = 0.15, ema_alpha: float = 0.3,
                 warmup_steps: int = 10, gpu_capacity_gb: float = 15.5,
                 sink_size: int = 4):
        super().__init__(threshold_gb=initial_threshold_gb,
                         page_size=page_size, sink_size=sink_size)
        self.safety_margin   = safety_margin
        self.ema_alpha       = ema_alpha
        self.warmup_steps    = warmup_steps
        self.gpu_capacity_gb = gpu_capacity_gb
        self._ema_gb         = None
        self._step           = 0

    def _update_threshold(self):
        if not torch.cuda.is_available():
            return
        cur = torch.cuda.memory_allocated(0) / (1024 ** 3)
        self._step += 1
        self._ema_gb = cur if self._ema_gb is None else \
            self.ema_alpha * cur + (1 - self.ema_alpha) * self._ema_gb
        if self._step > self.warmup_steps:
            adaptive = min(self._ema_gb * (1 + self.safety_margin),
                           self.gpu_capacity_gb * 0.92)
            self.threshold_bytes = adaptive * (1024 ** 3)

    def offload_kv_cache(self, past_key_values):
        self._update_threshold()
        return super().offload_kv_cache(past_key_values)

    def reset(self):
        super().reset()
        self._ema_gb = None
        self._step   = 0


class PredictiveMemoryPolicy(AdaptiveVirtualPageManager):
    # Defaults match Mistral-7B; calibrate_for_model() overrides at runtime.
    _num_kv_heads   = 8
    _head_dim       = 128
    _num_layers     = 32
    _bytes_per_elem = 2
    _kv_factor      = 2

    def calibrate_for_model(self, model):
        cfg = model.config
        self._num_kv_heads = getattr(cfg, "num_key_value_heads",
                             getattr(cfg, "num_attention_heads", 8))
        self._head_dim     = getattr(cfg, "head_dim",
                             cfg.hidden_size // getattr(cfg, "num_attention_heads", 1))
        self._num_layers   = getattr(cfg, "num_hidden_layers", 32)

    def predict_kv_increment_gb(self, delta_tokens: int) -> float:
        b = (delta_tokens * self._num_kv_heads * self._head_dim * self._num_layers
             * self._kv_factor * self._bytes_per_elem)
        return b / (1024 ** 3)

    def should_offload_predictive(self, delta_tokens: int = 1) -> bool:
        if not torch.cuda.is_available():
            return False
        cur = torch.cuda.memory_allocated(0) / (1024 ** 3)
        thr = self.threshold_bytes / (1024 ** 3)
        return (cur + self.predict_kv_increment_gb(delta_tokens)) > thr

    def auto_calibrate(self, baseline_gb: float, headroom_gb: float = 0.9):
        thr = baseline_gb + headroom_gb
        self.threshold_bytes = thr * (1024 ** 3)
        print(f"[Predictive] Threshold auto-calibrated: {thr:.2f} GB")

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(1):
            return past_key_values
        return VirtualPageManager.offload_kv_cache(self, past_key_values)


class RAGAwarePager(PredictiveMemoryPolicy):
    """Segment-aware pager: low-relevance passages preferred for eviction.
    Eviction restricted to PRE-decode boundary to bound RoPE position drift —
    StreamingLLM offload covers in-decode pressure."""
    class KVSegment:
        def __init__(self, seg_id, score, start_pos, end_pos, label="", turn_id=0):
            self.seg_id = seg_id
            self.score = score
            self.start_pos = start_pos
            self.end_pos = end_pos
            self.label = label
            self.turn_id = turn_id

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._segments    = []
        self._next_seg_id = 0
        self._evictions   = 0
        self.global_turn  = 0
        # [P0.1] Reserved spans (sys/question/few-shot) — never evictable.
        self._reserved_ranges = []

    def auto_calibrate(self, baseline_gb, headroom_gb=0.9):
        thr = baseline_gb + headroom_gb
        self.threshold_bytes = thr * (1024 ** 3)
        print(f"[RAGAware] Threshold: {baseline_gb:.2f}+{headroom_gb:.2f}={thr:.2f} GB")

    def reserve_range(self, start: int, end: int, label: str = ""):
        """[P0.1] Mark a token span as never-evictable (system/question/few-shot)."""
        if end > start:
            self._reserved_ranges.append((start, end, label))

    def _overlaps_reserved(self, start: int, end: int) -> bool:
        for s, e, _ in self._reserved_ranges:
            if start < e and end > s:
                return True
        return False

    def register_segment_abs(self, start: int, end: int, score: float,
                              label: str = "", turn_id: int = -1) -> int:
        """[P0.1] Register a passage segment by ABSOLUTE position (start, end)
        in the actual prefilled `input_ids` — eliminates the drifty
        re-tokenization heuristic. Asserts no overlap with reserved spans."""
        assert end > start, f"Invalid segment span [{start},{end})"
        if self._overlaps_reserved(start, end):
            raise AssertionError(
                f"Segment '{label}' [{start},{end}) overlaps reserved (sys/question) "
                f"range — would corrupt context if evicted.")
        seg = RAGAwarePager.KVSegment(
            seg_id=self._next_seg_id, score=score,
            start_pos=start, end_pos=end, label=label,
            turn_id=self.global_turn if turn_id == -1 else turn_id)
        self._segments.append(seg)
        self._next_seg_id += 1
        return seg.seg_id

    def offload_kv_cache(self, past_key_values):
        # [P0.5] During decode: ONLY StreamingLLM offload. Segment-aware
        # eviction is restricted to maybe_evict_pre_decode (positions stable).
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(1):
            return past_key_values
        return VirtualPageManager.offload_kv_cache(self, past_key_values)

    def maybe_evict_pre_decode(self, past_key_values):
        """Heaviest path: invoked once between prefill and decode.
        Drops the lowest-relevance passage if memory pressure remains."""
        if not self._segments or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(1):
            return past_key_values
        def temporal(s):
            lag = self.global_turn - s.turn_id
            return s.score * math.exp(-0.4 * lag)
        self._segments.sort(key=temporal)
        victim = self._segments.pop(0)
        return self._evict_segment(past_key_values, victim)

    def _evict_segment(self, past_key_values, victim):
        start, end = victim.start_pos, victim.end_pos
        key_cache, value_cache = _get_kv_tensors(past_key_values)
        new_keys, new_values = [], []
        for k, v in zip(key_cache, value_cache):
            seq = k.shape[-2]
            s, e = min(start, seq), min(end, seq)
            if s == e:
                new_keys.append(k); new_values.append(v); continue
            if s == 0:
                k_kept, v_kept = k[:, :, e:, :], v[:, :, e:, :]
            elif e >= seq:
                k_kept, v_kept = k[:, :, :s, :], v[:, :, :s, :]
            else:
                k_kept = torch.cat([k[:, :, :s, :], k[:, :, e:, :]], dim=2)
                v_kept = torch.cat([v[:, :, :s, :], v[:, :, e:, :]], dim=2)
            new_keys.append(k_kept.contiguous())
            new_values.append(v_kept.contiguous())
        evicted = min(end, key_cache[0].shape[-2]) - min(start, key_cache[0].shape[-2])
        # Shift indices of surviving segments and reserved ranges.
        remaining = []
        for s in self._segments:
            if s.start_pos >= end:
                s.start_pos -= evicted; s.end_pos -= evicted
            elif s.end_pos > start:
                s.end_pos = max(s.start_pos, s.end_pos - evicted)
            if s.end_pos > s.start_pos:
                remaining.append(s)
        self._segments = remaining
        new_reserved = []
        for rs, re_, lbl in self._reserved_ranges:
            if rs >= end:
                new_reserved.append((rs - evicted, re_ - evicted, lbl))
            elif re_ > start:
                new_reserved.append((rs, max(rs, re_ - evicted), lbl))
            else:
                new_reserved.append((rs, re_, lbl))
        self._reserved_ranges = new_reserved
        self._evictions += 1
        new_cache = _rebuild_cache_from_tensors(new_keys, new_values)
        if torch.cuda.is_available():
            torch.cuda.synchronize(); torch.cuda.empty_cache()
        if self._evictions <= 3:
            print(f"[RAGAware] Evict #{self._evictions} '{victim.label}' "
                  f"score={victim.score:.3f} (-{evicted} tok)")
        return new_cache

    def reset(self):
        super().reset()
        self._segments.clear()
        self._next_seg_id = 0
        self._evictions   = 0
        self._reserved_ranges = []

    @property
    def eviction_count(self) -> int:
        return self._evictions


# ─── Hybrid retrieval (BM25 + dense + RRF + LIM reorder) ──────────────────
def _tokenize_bm25(text: str):
    return re.findall(r"\w+", text.lower())

class HybridRetriever:
    """Sparse + dense retrieval fused via RRF (Cormack 2009), reranked by a
    cross-encoder, then reordered to mitigate Lost-in-the-Middle (Liu 2024)."""
    def __init__(self, collection, embed_model, rerank_model, top_k=10, top_rerank=3):
        self.collection  = collection
        self.embed       = embed_model
        self.rerank      = rerank_model
        self.top_k       = top_k
        self.top_rerank  = top_rerank
        self._docs       = None
        self._bm25       = None

    def _ensure_bm25(self):
        if self._bm25 is not None:
            return
        all_docs = self.collection.get(include=["documents"])["documents"]
        self._docs = all_docs
        self._bm25 = BM25Okapi([_tokenize_bm25(d) for d in all_docs])
        print(f"[HybridRetriever] BM25 index built over {len(all_docs)} docs.")

    def _dense_topk(self, query):
        v = self.embed.encode([query], normalize_embeddings=True)
        n = min(self.top_k, self.collection.count())
        r = self.collection.query(query_embeddings=v.tolist(), n_results=n)
        return r["documents"][0]

    def _bm25_topk(self, query):
        self._ensure_bm25()
        scores = self._bm25.get_scores(_tokenize_bm25(query))
        idx = np.argsort(scores)[::-1][: self.top_k]
        return [self._docs[i] for i in idx]

    @staticmethod
    def _rrf(rank_lists, k=HYBRID_RRF_K):
        fused = {}
        for ranking in rank_lists:
            for rank, doc in enumerate(ranking):
                fused[doc] = fused.get(doc, 0.0) + 1.0 / (k + rank + 1)
        return [d for d, _ in sorted(fused.items(), key=lambda x: -x[1])]

    @staticmethod
    def _lost_in_middle(items):
        """Place strongest items at start and end, weakest in middle."""
        if len(items) <= 2:
            return items
        out = [None] * len(items)
        left, right = 0, len(items) - 1
        for i, it in enumerate(items):
            if i % 2 == 0:
                out[left] = it; left += 1
            else:
                out[right] = it; right -= 1
        return out

    def retrieve(self, query):
        dense = self._dense_topk(query)
        bm25  = self._bm25_topk(query)
        fused = self._rrf([dense, bm25])[: self.top_k]
        if not fused:
            return []
        pairs  = [(query, d) for d in fused]
        scores = self.rerank.predict(pairs)
        ranked = sorted(zip(scores, fused), reverse=True)[: self.top_rerank]
        coupled = [(d, float(s)) for s, d in ranked]
        return self._lost_in_middle(coupled)


def crag_triage(docs_scores,
                 trust=CRAG_TRUST_THRESH,
                 nocontext=CRAG_NOCONTEXT_THRESH):
    """[P1.2] CRAG (Yan et al. 2024): 3-bucket triage on RAW cross-encoder
    score (NOT min-max normalized — that washes out the signal).
       - top1 >= trust         -> 'trust'      (use context as-is)
       - nocontext < top1 < trust -> 'ambiguous' (context shown, model warned)
       - top1 <= nocontext     -> 'no_context' (drop context, parametric only)
    """
    if not docs_scores:
        return docs_scores, "no_context"
    raw = [s for _, s in docs_scores]
    top1 = max(raw)
    if top1 >= trust:
        return docs_scores, "trust"
    if top1 <= nocontext:
        return docs_scores, "no_context"
    return docs_scores, "ambiguous"


def should_retrieve_adaptive(docs_scores, gate=ADAPTIVE_RETRIEVAL_GATE) -> bool:
    """[P1.1] Adaptive retrieval gate (Mallen 2023; Self-RAG, Asai 2024).
    Skip RAG entirely if the top-1 reranker score is below `gate` —
    parametric knowledge dominates on common factoids.

    [Fase 2] `gate` é parametrizável: passe um threshold calibrado por
    `calibrate_adaptive_gate` para abrir RAG em ~50 % das queries."""
    if not docs_scores:
        return False
    top1 = max(s for _, s in docs_scores)
    return top1 >= gate


def calibrate_adaptive_gate(pre_fetched_scores, target_fire_rate=ADAPTIVE_GATE_TARGET_FIRE):
    """[Fase 2] Calibração empírica do gate: define o threshold como o
    percentil tal que `target_fire_rate` das queries fiquem acima dele.
    Aproxima a heurística de Mallen 2023 (popularidade ↔ confiança do
    retriever) sem depender de meta-data de popularidade.

    Retorna (threshold, fire_rate_estimado, n_queries)."""
    top1s = []
    for _q, ds in pre_fetched_scores.items():
        if ds:
            top1s.append(max(s for _, s in ds))
    if not top1s:
        return ADAPTIVE_RETRIEVAL_GATE, 0.0, 0
    top1s_sorted = sorted(top1s)
    # Fire rate desejada: queries cujos top1 >= threshold.
    # Threshold = percentil (1 - target_fire_rate).
    pct = max(0.0, min(1.0, 1.0 - target_fire_rate))
    idx = max(0, min(len(top1s_sorted) - 1, int(pct * len(top1s_sorted))))
    thr = top1s_sorted[idx]
    fire = sum(1 for s in top1s if s >= thr) / len(top1s)
    return thr, fire, len(top1s)


# ─── Few-shot ICL ─────────────────────────────────────────────────────────
class FewShotPool:
    def __init__(self, embed_model, k=FEWSHOT_K):
        self.embed = embed_model
        self.k     = k
        self.examples = []   # (q, a, embedding)

    def add(self, q, gold_a):
        v = self.embed.encode([q], normalize_embeddings=True)[0]
        self.examples.append((q, gold_a, v))

    def select(self, query):
        if not self.examples:
            return []
        v = self.embed.encode([query], normalize_embeddings=True)[0]
        scored = [(float(np.dot(v, x[2])), x) for x in self.examples]
        scored.sort(reverse=True)
        return [(q, a) for _, (q, a, _) in scored[: self.k]]


# ─── Engine ───────────────────────────────────────────────────────────────
class HardenedLLMEngine:
    def __init__(self, model_name, vector_db_path, paging_manager=None,
                 top_k_retrieval=10, top_k_rerank=3, fewshot_pool=None):
        self.model_name      = model_name
        self.vector_db_path  = vector_db_path
        self.pager           = paging_manager
        self.top_k_retrieval = top_k_retrieval
        self.top_k_rerank    = top_k_rerank
        self.model           = None
        self.tokenizer       = None
        self.collection      = None
        self._embed_model    = None
        self._rerank_model   = None
        self.retriever       = None
        self.fewshot         = fewshot_pool
        self.system_msg = ("You are a precise factual assistant. "
                           "Answer the question using ONLY the provided context. "
                           "Give a SHORT, DIRECT answer in 1 to 5 words. "
                           "Do not explain or elaborate.")
        self._gen_config_fast = None
        self._offload_stream  = None
        # [Fase 1.2] Asserções e telemetria do caminho paginado.
        self._enable_pager_asserts  = True
        self._pager_assert_failures = 0
        self._pager_prefill_drift   = 0

    def initialize_runtime(self):
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.35, device=0)
            print("[Engine] PyTorch alloc capped at 35% (~5.4GB on T4-16GB).")
            self._offload_stream = torch.cuda.Stream()
        client = chromadb.PersistentClient(path=self.vector_db_path)
        self.collection = client.get_collection("cognito_knowledge_base")

        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.float16,
                                  bnb_4bit_use_double_quant=True)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name, quantization_config=bnb, device_map="auto",
                torch_dtype=torch.float16, attn_implementation="sdpa")
            print("[Engine] attn_implementation=sdpa active.")
        except Exception:
            print("[Engine] SDPA unavailable, default attention.")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name, quantization_config=bnb, device_map="auto",
                torch_dtype=torch.float16)
        self.model.eval()

        if isinstance(self.pager, PredictiveMemoryPolicy):
            self.pager.calibrate_for_model(self.model)

        if self.pager is not None and hasattr(self.pager, "auto_calibrate"):
            base = torch.cuda.memory_allocated(0) / (1024 ** 3)
            self.pager.auto_calibrate(base, headroom_gb=0.9)

        self._embed_model  = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        self._rerank_model = CrossEncoder(RERANKER_MODEL_NAME)
        self.retriever     = HybridRetriever(self.collection, self._embed_model,
                                              self._rerank_model,
                                              top_k=self.top_k_retrieval,
                                              top_rerank=self.top_k_rerank)
        if self.fewshot is None:
            self.fewshot = FewShotPool(self._embed_model)

        # Fast-path: prompt-lookup decoding (Saxena 2023).
        gc_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            prompt_lookup_num_tokens=PROMPT_LOOKUP_NUM_TOKENS,
        )
        if USE_HQQ_KV_CACHE:
            try:
                gc_kwargs["cache_implementation"] = "quantized"
                gc_kwargs["cache_config"] = {"backend": "HQQ", "nbits": 4,
                                              "compute_dtype": torch.float16,
                                              "device": "cuda"}
                print("[Engine] HQQ INT4 KV-cache enabled (fast path).")
            except Exception as e:
                print(f"[Engine] HQQ KV-cache unavailable: {e}")
        self._gen_config_fast = GenerationConfig(**gc_kwargs)


    def retrieve_with_scores(self, query: str) -> list:
        if self.retriever is None:
            return []
        return self.retriever.retrieve(query)

    # ── [P0.1] Piece-based prompt construction with exact span tracking ──
    def _build_prompt_pieces(self, query, docs_scores, fewshot_pairs=None,
                              crag_label="trust"):
        """Returns (pieces, passage_indices) where:
        - pieces: list of (label, text) — concatenation forms the full prompt
        - passage_indices: indices of pieces that are RAG passages (evictable)
        Mistral-style chat template: [INST] ... [/INST] markers wrap user msg.
        """
        sys_text = self.system_msg
        if crag_label == "ambiguous":
            sys_text += " If the context is not relevant, respond with UNKNOWN."
        elif crag_label == "no_context":
            sys_text = (self.system_msg +
                         " You have no context available. Respond with UNKNOWN.")

        # Mistral-Instruct template: <s>[INST] {body} [/INST]
        template_open  = "<s>[INST] "
        template_close = " [/INST]"

        body_pieces = []
        body_pieces.append(("sys", sys_text + "\n\n"))
        if fewshot_pairs:
            for i, (q, a) in enumerate(fewshot_pairs):
                body_pieces.append((f"fs{i}", f"Example Q: {q}\nExample A: {a}\n\n"))
        passage_idx = []
        if docs_scores and crag_label != "no_context":
            body_pieces.append(("ctx_intro", "Context:\n"))
            for i, (d, _) in enumerate(docs_scores):
                idx = len(body_pieces)
                body_pieces.append((f"p{i}", f"[{i+1}] {d}\n\n"))
                passage_idx.append(idx)
        body_pieces.append(("question", f"Question: {query}"))

        pieces = [("tmpl_open", template_open)] + body_pieces + [("tmpl_close", template_close)]
        passage_idx_in_pieces = [i + 1 for i in passage_idx]  # +1 for tmpl_open
        return pieces, passage_idx_in_pieces

    def _tokenize_pieces(self, pieces):
        """Tokenize each piece WITHOUT adding special tokens, recording exact
        (start, end) spans in the concatenated input_ids."""
        device = next(self.model.parameters()).device
        all_ids = []
        spans = []
        cursor = 0
        for label, text in pieces:
            if not text:
                spans.append((label, cursor, cursor))
                continue
            ids = self.tokenizer(text, return_tensors="pt",
                                  add_special_tokens=False)["input_ids"][0]
            n = ids.shape[-1]
            spans.append((label, cursor, cursor + n))
            all_ids.append(ids)
            cursor += n
        full = torch.cat(all_ids, dim=0).unsqueeze(0).to(device) if all_ids \
               else torch.empty((1, 0), dtype=torch.long, device=device)
        return full, spans

    def _chunked_prefill(self, full_input_ids, past_kv=None,
                          chunk_size=CHUNKED_PREFILL_CHUNK_SIZE):
        """Chunked prefill (Sarathi-Serve OSDI 2024) with explicit cache_position."""
        device  = full_input_ids.device
        n_total = full_input_ids.shape[-1]
        cur_len = 0 if past_kv is None else get_cache_seq_len(past_kv)
        for start in range(0, n_total, chunk_size):
            end   = min(start + chunk_size, n_total)
            chunk = full_input_ids[:, start:end]
            chunk_n = chunk.shape[-1]
            mask  = torch.ones((1, cur_len + chunk_n), dtype=torch.long, device=device)
            cache_position = torch.arange(cur_len, cur_len + chunk_n, device=device)
            with torch.no_grad():
                out = self.model(input_ids=chunk, attention_mask=mask,
                                  past_key_values=past_kv,
                                  cache_position=cache_position, use_cache=True)
            past_kv = out.past_key_values
            cur_len += chunk_n
            # No offload during prefill: truncation breaks RoPE absolute-position contract.
        return past_kv, cur_len

    def _passage_level_prefill(self, docs_scores, query, fewshot_pairs=None,
                                crag_label="trust"):
        """[P0.1] Build prompt by pieces, tokenize each piece individually
        (so passage spans align with actual KV cache positions), prefill in
        chunks, then register segments via register_segment_abs.

        [Fase 1.2] Inclui asserção de equivalência tokenização-piecewise vs
        tokenização-monolítica para travar regressão silenciosa: a única
        forma de C6 corromper geração com `Paged:0` e sem eviction é se os
        spans não baterem com o input_ids de fato consumido pelo modelo."""
        if self.pager is not None:
            self.pager.reset(); self.pager.active = True

        pieces, passage_idx = self._build_prompt_pieces(
            query, docs_scores, fewshot_pairs, crag_label)
        full_ids, spans = self._tokenize_pieces(pieces)
        n_total = full_ids.shape[-1]

        # [Fase 1.2] Asserção forte: tokenização piecewise == monolítica.
        # Rationale: se diferem, _build_prompt_pieces produz spans que NÃO
        # correspondem ao input_ids real consumido pelo `_chunked_prefill`,
        # invalidando register_segment_abs e — mais grave — possivelmente
        # quebrando RoPE positions em re-tokenização interna.
        if self._enable_pager_asserts:
            full_text = "".join(t for _, t in pieces)
            mono = self.tokenizer(full_text, return_tensors="pt",
                                   add_special_tokens=False)["input_ids"][0]
            piece_ids = full_ids[0].cpu()
            mono_ids  = mono.cpu()
            if not torch.equal(piece_ids, mono_ids):
                # Não levantar — apenas alertar e seguir, para não quebrar
                # avaliação. Mas registrar a divergência em telemetria.
                self._pager_assert_failures += 1
                if self._pager_assert_failures <= 3:
                    print(f"[Pager-Assert] piecewise≠mono tokenization: "
                          f"piece_len={piece_ids.numel()} mono_len={mono_ids.numel()}")

        # Reserve sys / few-shot / question / template markers as never-evictable.
        if isinstance(self.pager, RAGAwarePager):
            for label, s, e in spans:
                if label.startswith("p"):  # passage
                    continue
                # Reserve everything else (sys, fs*, ctx_intro, question, tmpl_*)
                if e > s:
                    self.pager.reserve_range(s, e, label=label)

        # Prefill all-but-last token in chunks (last token is fed during decode).
        past_kv, cur_len = self._chunked_prefill(full_ids[:, :-1], past_kv=None)

        # [Fase 1.2] Telemetria por query: garante que cur_len após prefill
        # bate com n_total - 1 (último token vai no decode). Divergência
        # indica eviction prematura ou bug em _chunked_prefill.
        expected_after_prefill = max(0, n_total - 1)
        if cur_len != expected_after_prefill:
            self._pager_prefill_drift += 1
            if self._pager_prefill_drift <= 3:
                print(f"[Pager-Drift] cur_len={cur_len} expected={expected_after_prefill} "
                      f"n_total={n_total} segments={len(getattr(self.pager, '_segments', []))}")

        # Register passages with EXACT positions from spans.
        if isinstance(self.pager, RAGAwarePager) and docs_scores and \
           crag_label != "no_context":
            raw = [sc for _, sc in docs_scores]
            mn, mx = min(raw), max(raw)
            normed = [(s - mn) / (mx - mn) if mx > mn else 1.0 for s in raw]
            score_iter = iter(normed)
            for i_piece in passage_idx:
                label, s, e = spans[i_piece]
                # Only register if entirely within the prefilled range.
                if e <= cur_len and e > s:
                    score = next(score_iter, 0.0)
                    self.pager.register_segment_abs(start=s, end=e, score=score,
                                                     label=label)
                else:
                    next(score_iter, None)
            past_kv = self.pager.maybe_evict_pre_decode(past_kv)
            cur_len = get_cache_seq_len(past_kv)

        last_tok = full_ids[0, -1]
        return past_kv, cur_len, last_tok, full_ids

    def _generate_token_by_token(self, input_ids, max_new_tokens,
                                   initial_past_kv=None, cur_cache_len=None,
                                   t_prefill_start=None):
        """[P0.3] If `t_prefill_start` is provided, TTFT = t_first_token -
        t_prefill_start (honest, includes prefill). Otherwise TTFT = decode
        first-token latency only (legacy semantics)."""
        device = input_ids.device
        past_kv = initial_past_kv
        generated = input_ids.clone()
        if self.pager:
            self.pager.active = True
        local_cache_len = cur_cache_len if cur_cache_len is not None else 0
        ttft = 0.0
        decode_t = []
        t0 = t_prefill_start if t_prefill_start is not None else time.time()
        with torch.no_grad():
            for step in range(max_new_tokens):
                ts = time.time()
                model_input = generated[:, -1:] if (step > 0 or past_kv is not None) else generated
                cur_n = model_input.shape[-1]
                cur_mask = torch.ones((1, local_cache_len + cur_n),
                                        dtype=torch.long, device=device)
                # [P0.2] cache_position required when reusing past_kv.
                if past_kv is not None:
                    cache_position = torch.arange(local_cache_len,
                                                   local_cache_len + cur_n,
                                                   device=device)
                    out = self.model(input_ids=model_input, attention_mask=cur_mask,
                                      past_key_values=past_kv,
                                      cache_position=cache_position, use_cache=True)
                else:
                    out = self.model(input_ids=model_input, attention_mask=cur_mask,
                                      use_cache=True)
                next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                past_kv  = out.past_key_values
                local_cache_len += cur_n
                # In-decode offload removed: breaks RoPE positions. Eviction only pre-decode.
                generated = torch.cat([generated, next_tok], dim=-1)
                te = time.time()
                if step == 0:
                    ttft = te - t0
                else:
                    decode_t.append(te - ts)
                if next_tok.item() == self.tokenizer.eos_token_id:
                    break
        itl = sum(decode_t) / len(decode_t) if decode_t else 0.0
        return generated, ttft, itl

    def _build_prompt_legacy(self, query, context_text, fewshot_pairs=None,
                              crag_label="trust", use_query_repetition=False):
        """Used only by the standard (non-passage-prefill) path."""
        sys_text = self.system_msg
        if crag_label == "ambiguous":
            sys_text += " If the context is not relevant, respond with UNKNOWN."
        elif crag_label == "no_context":
            sys_text = (self.system_msg +
                         " You have no context available. Respond with UNKNOWN.")
        body = sys_text + "\n\n"
        if fewshot_pairs:
            for q, a in fewshot_pairs:
                body += f"Example Q: {q}\nExample A: {a}\n\n"
        if context_text and crag_label != "no_context":
            body += f"Context:\n{context_text}\n\n"
        body += f"Question: {query}"
        if use_query_repetition:
            body += f"\n\nImportant: carefully read the context above and answer: {query}"
        msgs = [{"role": "user", "content": body}]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False,
                                                    add_generation_prompt=True)

    def _generate_hybrid(self, input_ids, attention_mask, max_new_tokens,
                          use_paging, t_prefill_start, initial_past_kv=None,
                          cur_cache_len=None):
        seq_len = input_ids.shape[-1] if cur_cache_len is None else cur_cache_len
        if not use_paging or self.pager is None or seq_len < PAGING_CONTEXT_THRESHOLD_TOKENS:
            if self.pager:
                self.pager.reset(); self.pager.active = False
            with torch.no_grad():
                try:
                    gen = self.model.generate(input_ids=input_ids,
                                                attention_mask=attention_mask,
                                                past_key_values=initial_past_kv,
                                                generation_config=self._gen_config_fast)
                except Exception as e:
                    print(f"[Engine] fast-path failed ({type(e).__name__}); vanilla fallback.")
                    gen = self.model.generate(input_ids=input_ids,
                                                attention_mask=attention_mask,
                                                past_key_values=initial_past_kv,
                                                max_new_tokens=max_new_tokens, use_cache=True,
                                                pad_token_id=self.tokenizer.pad_token_id)
            lat = time.time() - t_prefill_start
            n_new = gen.shape[-1] - input_ids.shape[-1]
            # Approximate TTFT for fast-path (model.generate is opaque): heuristic
            # split — first token tends to dominate when n_new is small. Honest
            # bound: ttft <= total latency.
            approx_ttft = lat if n_new <= 1 else lat * (1.0 / max(1, n_new)) + 0.6 * lat
            approx_ttft = min(approx_ttft, lat)
            approx_itl = max(0.0, (lat - approx_ttft) / max(1, n_new - 1))
            return gen, "fast_path", approx_ttft, approx_itl
        gen, ttft, itl = self._generate_token_by_token(
            input_ids, max_new_tokens,
            initial_past_kv=initial_past_kv, cur_cache_len=cur_cache_len,
            t_prefill_start=t_prefill_start)
        return gen, "paging_path", ttft, itl

    def generate(self, query: str, retrieved_context: str = "", use_rag: bool = True,
                  max_new_tokens: int = MAX_NEW_TOKENS, use_paging: bool = True,
                  use_chunked_prefill: bool = False, use_prefix_caching: bool = False,
                  use_passage_prefill: bool = False, docs_scores: list = None,
                  use_fewshot: bool = False, use_crag: bool = False,
                  use_adaptive_gate: bool = False,
                  use_query_repetition: bool = False,
                  use_context_reorder: bool = False,
                  use_context_trim: bool = False) -> dict:
        device = next(self.model.parameters()).device
        fewshot = self.fewshot.select(query) if (use_fewshot and self.fewshot) else None
        crag_label = "trust"
        if use_crag and (docs_scores is not None):
            _, crag_label = crag_triage(docs_scores)

        # [P1.1] Adaptive retrieval gate: skip RAG entirely if reranker top-1
        # below threshold (parametric knowledge dominates on common factoids).
        adaptive_skipped = False
        if use_adaptive_gate and use_rag and docs_scores is not None:
            if not should_retrieve_adaptive(docs_scores):
                use_rag = False
                use_passage_prefill = False
                docs_scores = []
                adaptive_skipped = True
                crag_label = "no_context"

        # Timer starts before prefill for honest TTFT measurement.
        torch.cuda.reset_peak_memory_stats()
        t_prefill_start = time.time()

        # Path A: passage-level prefill (RAGAwarePager)
        if use_passage_prefill and use_rag and docs_scores:
            try:
                past_kv, cur_len, last_tok, full_ids = self._passage_level_prefill(
                    docs_scores, query, fewshot_pairs=fewshot, crag_label=crag_label)
                tail = last_tok.unsqueeze(0).unsqueeze(0)
                gen, ttft, itl = self._generate_token_by_token(
                    input_ids=tail, max_new_tokens=max_new_tokens,
                    initial_past_kv=past_kv, cur_cache_len=cur_len,
                    t_prefill_start=t_prefill_start)
                status = "SUCCESS"
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    force_cleanup()
                    return {"response": "", "latency_sec": 0, "throughput_tps": 0,
                            "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
                            "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE",
                            "ttft_sec": 0.0, "itl_sec": 0.0,
                            "eviction_events": 0, "streamllm_offloads": 0,
                            "adaptive_skipped": adaptive_skipped}
                raise
            lat = time.time() - t_prefill_start
            n_new = gen.shape[-1] - 1
            response = self.tokenizer.decode(gen[0, 1:], skip_special_tokens=True)
            ev = (self.pager.eviction_count
                  if isinstance(self.pager, RAGAwarePager) else 0)
            sllm = (self.pager.streamllm_offloads if self.pager else 0)
            return {"response": response, "latency_sec": lat,
                    "throughput_tps": n_new / lat if lat > 0 else 0.0,
                    "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
                    "blocks_paged": self.pager.blocks_in_cpu if self.pager else 0,
                    "gen_path": "passage_prefill", "ttft_sec": ttft, "itl_sec": itl,
                    "status": status, "eviction_events": ev,
                    "streamllm_offloads": sllm,
                    "adaptive_skipped": adaptive_skipped}

        # Path B: standard prompt
        ctx_text = retrieved_context
        # [LitM Camada 3] BM25-based trimming: reduces 16k/32k → 8k
        if use_context_trim and ctx_text and self.tokenizer:
            ctx_text = trim_context_bm25(ctx_text, query, self.tokenizer,
                                          max_tokens=8192)
        # [LitM Camada 2] Lost-in-Middle reordering (without trimming)
        elif use_context_reorder and ctx_text and self.tokenizer:
            ctx_text = reorder_context_lim(ctx_text, query, self.tokenizer)
        if len(ctx_text) > MAX_CONTEXT_CHARS:
            ctx_text = ctx_text[:MAX_CONTEXT_CHARS] + "... [TRUNCATED]"
        prompt = self._build_prompt_legacy(query, ctx_text if use_rag else "",
                                              fewshot, crag_label,
                                              use_query_repetition=use_query_repetition)
        enc = self.tokenizer(prompt, return_tensors="pt",
                              add_special_tokens=False).to(device)
        input_ids, attention_mask = enc["input_ids"], enc["attention_mask"]
        try:
            if use_chunked_prefill and input_ids.shape[-1] >= CHUNKED_PREFILL_THRESHOLD_TOKENS:
                # Pager activation gated on use_paging so L2 runs as clean baseline.
                if self.pager and use_paging:
                    self.pager.reset(); self.pager.active = True
                elif self.pager:
                    self.pager.reset(); self.pager.active = False
                past_kv, cur_len = self._chunked_prefill(input_ids[:, :-1], past_kv=None)
                # Register coarse ctx segment; reserve last 64 tokens (instruction tail).
                n_pre = input_ids[:, :-1].shape[-1]
                tail_reserve = min(64, n_pre)
                if (use_paging and isinstance(self.pager, RAGAwarePager)
                        and n_pre > tail_reserve + 32):
                    # Segment-aware: register coarse context, evict if under pressure
                    self.pager.reserve_range(n_pre - tail_reserve, n_pre,
                                              label="instr_tail")
                    self.pager.register_segment_abs(
                        start=0, end=n_pre - tail_reserve,
                        score=1.0, label="ctx_coarse")
                    past_kv = self.pager.maybe_evict_pre_decode(past_kv)
                    cur_len = get_cache_seq_len(past_kv)
                elif (use_paging and isinstance(self.pager, H2OEvictionPolicy)):
                    # H2O: evict by L2-norm importance if VRAM > threshold
                    past_kv = self.pager.maybe_evict_pre_decode(past_kv)
                    cur_len = get_cache_seq_len(past_kv)
                tail = input_ids[:, -1:]
                gen_dec, ttft, itl = self._generate_token_by_token(
                    input_ids=tail, max_new_tokens=max_new_tokens,
                    initial_past_kv=past_kv, cur_cache_len=cur_len,
                    t_prefill_start=t_prefill_start)
                gen      = torch.cat([input_ids[:, :-1], gen_dec], dim=-1)
                gen_path = "chunked_prefill"
            else:
                gen, gen_path, ttft, itl = self._generate_hybrid(
                    input_ids, attention_mask, max_new_tokens, use_paging,
                    t_prefill_start, initial_past_kv=None)
            status = "SUCCESS"
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                force_cleanup()
                return {"response": "", "latency_sec": 0, "throughput_tps": 0,
                        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
                        "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE",
                        "ttft_sec": 0.0, "itl_sec": 0.0,
                        "eviction_events": 0, "streamllm_offloads": 0,
                        "adaptive_skipped": adaptive_skipped}
            raise
        lat = time.time() - t_prefill_start
        n_new = gen.shape[-1] - input_ids.shape[-1]
        response = self.tokenizer.decode(gen[0, input_ids.shape[-1]:], skip_special_tokens=True)
        ev = (self.pager.eviction_count
              if isinstance(self.pager, RAGAwarePager) else 0)
        sllm = (self.pager.streamllm_offloads if self.pager else 0)
        return {"response": response, "latency_sec": lat,
                "throughput_tps": n_new / lat if lat > 0 else 0.0,
                "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
                "blocks_paged": self.pager.blocks_in_cpu if self.pager else 0,
                "gen_path": gen_path, "ttft_sec": ttft, "itl_sec": itl,
                "status": status, "eviction_events": ev,
                "streamllm_offloads": sllm,
                "adaptive_skipped": adaptive_skipped}


# ─── Metrics ─────────────────────────────────────────────────────────────
def normalize_answer(s):
    return _normalize_text(s)
def exact_match_score(pred, gt):
    # SQuAD-canonical EM: strict equality after normalize (substring inflates scores).
    return normalize_answer(pred) == normalize_answer(gt)
def f1_score(pred, gt):
    p, t = normalize_answer(pred).split(), normalize_answer(gt).split()
    if not p or not t: return 0.0
    common = Counter(p) & Counter(t); n = sum(common.values())
    if n == 0: return 0.0
    pr, rc = n / len(p), n / len(t)
    return (2 * pr * rc) / (pr + rc)
def metric_max_over_ground_truths(fn, pred, truths):
    return max(fn(pred, gt) for gt in truths) if truths else 0.0


def bootstrap_ci(values, n_resamples=BOOTSTRAP_RESAMPLES, seed=13):
    """Percentile bootstrap (Efron 1979): returns (mean, lo95, hi95)."""
    if not values:
        return (0.0, 0.0, 0.0)
    rng = random.Random(seed)
    n = len(values)
    samples = []
    for _ in range(n_resamples):
        s = sum(values[rng.randint(0, n - 1)] for _ in range(n)) / n
        samples.append(s)
    samples.sort()
    mean = sum(values) / n
    lo = samples[int(0.025 * n_resamples)]
    hi = samples[int(0.975 * n_resamples)]
    return mean, lo, hi


def wilson_ci(successes: int, n: int, z: float = 1.96):
    """[P3.3] Wilson score interval for a binomial proportion (Brown 2001).
    More accurate than normal-approx CI for small n / extreme p; sanity-check
    cross-validation against the bootstrap percentile CI."""
    if n == 0:
        return 0.0, 0.0, 0.0
    p = successes / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return p, max(0.0, center - half), min(1.0, center + half)


def paired_bootstrap_ci(values_a, values_b, n_resamples=BOOTSTRAP_RESAMPLES, seed=13):
    """[Fase 5.3] Paired bootstrap (Efron & Tibshirani 1993, §16.3) sobre a
    diferença Δ = mean(a) - mean(b), reamostrando o índice da query (não os
    valores independentemente). Preserva a correlação intra-query entre
    configurações, gerando CI mais estreito e p-value bilateral honesto.

    Hipotese nula H0: Δ = 0. Reportar p (two-sided) = 2·min(P(Δ*≥0), P(Δ*≤0))
    sob reamostragem com troca dos pares.

    Retorna (delta_mean, lo95, hi95, p_two_sided)."""
    assert len(values_a) == len(values_b), \
        "paired bootstrap requer pares alinhados (mesma query, configs distintas)"
    n = len(values_a)
    if n == 0:
        return 0.0, 0.0, 0.0, 1.0
    diffs = [a - b for a, b in zip(values_a, values_b)]
    delta = sum(diffs) / n
    rng = random.Random(seed)
    samples = []
    for _ in range(n_resamples):
        idxs = [rng.randint(0, n - 1) for _ in range(n)]
        s = sum(diffs[i] for i in idxs) / n
        samples.append(s)
    samples.sort()
    lo = samples[int(0.025 * n_resamples)]
    hi = samples[int(0.975 * n_resamples)]
    # p-value bilateral via reamostragem centrada em zero (achievable shift).
    centered = [s - delta for s in samples]
    p_ge = sum(1 for s in centered if s >= abs(delta)) / n_resamples
    p_le = sum(1 for s in centered if s <= -abs(delta)) / n_resamples
    p_two = min(1.0, p_ge + p_le)
    return delta, lo, hi, p_two


# ── [P3.1] Retrieval-side metrics (Karpukhin 2020 / DPR) ─────────────────
def _alias_in_doc(aliases, doc_lower):
    for a in aliases:
        a_norm = re.sub(r"\s+", " ", a.strip().lower())
        if len(a_norm) >= 2 and a_norm in doc_lower:
            return True
    return False

def retrieval_metrics(pre_fetched_scores, gold_answers, k=3, gold_hits=None):
    """Return dict with recall@k, MRR, gold_hit_rate (= recall@k since k=top_rerank).
    `pre_fetched_scores`: {query: [(doc, score), ...]}.
    `gold_answers`: {query: [aliases]}. Substring match on lowercased text.

    [Fase 0.1] Se `gold_hits` (gerado em ingestão por P1.3) for fornecido,
    restringe o cálculo ao subset retrievable (`gold_hits[q] == True`),
    isolando qualidade do retriever da cobertura do corpus.
    Retorna `conditional_recall@k` em vez de `recall@k` cego."""
    n = 0
    hits = 0
    rr_sum = 0.0
    for q, ds in pre_fetched_scores.items():
        aliases = gold_answers.get(q) or []
        if not aliases:
            continue
        if gold_hits is not None and not gold_hits.get(q, False):
            continue
        n += 1
        topk = ds[:k]
        rank_first = None
        for r, (d, _) in enumerate(topk, start=1):
            if _alias_in_doc(aliases, d.lower()):
                if rank_first is None:
                    rank_first = r
        if rank_first is not None:
            hits += 1
            rr_sum += 1.0 / rank_first
    if n == 0:
        return {"recall@k": 0.0, "MRR": 0.0, "n": 0,
                "conditional": gold_hits is not None}
    return {"recall@k": hits / n, "MRR": rr_sum / n, "n": n, "k": k,
            "conditional": gold_hits is not None}


# ─── Eval orchestrator ───────────────────────────────────────────────────
def run_evaluation(queries, configs, engine, gold_answers,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pre_fetched=None, pre_fetched_scores=None,
                    gold_hits=None, verbose=None,
                    trace_tag="eval"):
    if pre_fetched is None:
        print("\n[Eval] Pre-fetching retrieval (hybrid BM25+dense+RRF, LIM reorder)...")
        pre_fetched, pre_fetched_scores = {}, {}
        for q in queries:
            ds = engine.retrieve_with_scores(q)
            pre_fetched_scores[q] = ds
            pre_fetched[q] = "\n".join([d for d, _ in ds])

    # [P3.1] Retrieval-side metrics — printed BEFORE eval, while embed/rerank
    # are still in scope; reported once per session (retrieval is shared).
    if pre_fetched_scores and gold_answers:
        rm3 = retrieval_metrics(pre_fetched_scores, gold_answers, k=3)
        rm10 = retrieval_metrics(pre_fetched_scores, gold_answers, k=10)
        print(f"[Retrieval@3]  recall={rm3['recall@k']*100:.1f}% MRR={rm3['MRR']:.3f} (n={rm3['n']})")
        print(f"[Retrieval@10] recall={rm10['recall@k']*100:.1f}% MRR={rm10['MRR']:.3f} (n={rm10['n']})")
        # [Fase 0.1] Conditional recall: isola qualidade do retriever da
        # cobertura do corpus (decisão crítica conforme plano Fase 0.2).
        if gold_hits:
            cm3 = retrieval_metrics(pre_fetched_scores, gold_answers, k=3,
                                     gold_hits=gold_hits)
            cm10 = retrieval_metrics(pre_fetched_scores, gold_answers, k=10,
                                      gold_hits=gold_hits)
            print(f"[ConditionalRetrieval@3]  recall={cm3['recall@k']*100:.1f}% "
                  f"MRR={cm3['MRR']:.3f} (n={cm3['n']}, subset retrievable)")
            print(f"[ConditionalRetrieval@10] recall={cm10['recall@k']*100:.1f}% "
                  f"MRR={cm10['MRR']:.3f} (n={cm10['n']})")
            if cm3['recall@k'] < 0.6 and rm3['recall@k'] < 0.4:
                print("  ⚠ conditional_recall@3 < 0.6: retriever fraco MESMO no "
                      "subset com gold no corpus — considerar NV-Embed-v2 ou "
                      "expandir top_k de retrieval.")

    if engine._embed_model:
        del engine._embed_model; engine._embed_model = None
    if engine._rerank_model:
        del engine._rerank_model; engine._rerank_model = None
    engine.retriever = None
    force_cleanup()

    if verbose is None:
        verbose = EVAL_VERBOSE
    trace_fp = open(EVAL_TRACE_PATH, "a", encoding="utf-8")
    all_results = {}
    try:
        for cfg in configs:
            label = cfg["label"]
            print(f"\n[{trace_tag}] === {label} ===")
            if cfg.get("use_paging") and cfg.get("threshold_gb") and engine.pager:
                engine.pager.threshold_bytes = cfg["threshold_gb"] * (1024 ** 3)
            records = []
            ok = oom = 0
            for i, q in enumerate(queries):
                m = engine.generate(query=q,
                                      retrieved_context=pre_fetched.get(q, ""),
                                      use_rag=cfg.get("use_rag", True),
                                      max_new_tokens=max_new_tokens,
                                      use_paging=cfg.get("use_paging", False),
                                      use_chunked_prefill=cfg.get("use_chunked_prefill", False),
                                      use_prefix_caching=cfg.get("use_prefix_caching", False),
                                      use_passage_prefill=cfg.get("use_passage_prefill", False),
                                      use_fewshot=cfg.get("use_fewshot", False),
                                      use_crag=cfg.get("use_crag", False),
                                      use_adaptive_gate=cfg.get("use_adaptive_gate", False),
                                      use_query_repetition=cfg.get("use_query_repetition", False),
                                      use_context_reorder=cfg.get("use_context_reorder", False),
                                      use_context_trim=cfg.get("use_context_trim", False),
                                      docs_scores=pre_fetched_scores.get(q, []))
                m["em_score"] = (metric_max_over_ground_truths(exact_match_score, m["response"], gold_answers.get(q, []))
                                  if m["status"] == "SUCCESS" else None)
                m["f1_tok"] = (metric_max_over_ground_truths(f1_score, m["response"], gold_answers.get(q, []))
                                if m["status"] == "SUCCESS" else None)
                m["query"] = q
                records.append(m)
                # [P0] persist full trace; skip large fields for size
                trace = {"tag": trace_tag, "cfg": label, "qid": i, "q": q[:120],
                         "benchmark": trace_tag,
                         "status": m["status"],
                         "em": m["em_score"], "f1": m["f1_tok"],
                         "vram_gb": round(m.get("peak_vram_gb", 0.0), 3),
                         "ttft_s": round(m.get("ttft_sec", 0.0), 3),
                         "itl_ms": round(m.get("itl_sec", 0.0) * 1000, 1),
                         "paged": m.get("blocks_paged", 0),
                         "evicts": m.get("eviction_events", 0),
                         "sllm": m.get("streamllm_offloads", 0),
                         "gated_off": bool(m.get("adaptive_skipped", False)),
                         "resp": (m.get("response") or "")[:200]}
                trace_fp.write(json.dumps(trace, ensure_ascii=False) + "\n")
                if m["status"] == "SUCCESS":
                    ok += 1
                else:
                    oom += 1
                # Conditional console output
                show = verbose or (m["status"] != "SUCCESS") or \
                       (i == 0) or ((i + 1) % EVAL_PROGRESS_EVERY == 0) or \
                       (i + 1 == len(queries))
                if show:
                    extra = ""
                    if m.get("adaptive_skipped"):
                        extra += " gated"
                    if m.get("eviction_events", 0) > 0:
                        extra += f" evict={m['eviction_events']}"
                    print(f"  [{i+1:>3}/{len(queries)}] {m['status'][:4]} "
                          f"EM={m['em_score']} F1={(m['f1_tok'] or 0):.2f} "
                          f"vram={m.get('peak_vram_gb',0):.2f}GB "
                          f"ttft={m.get('ttft_sec',0):.2f}s "
                          f"itl={m.get('itl_sec',0)*1000:.0f}ms{extra}")
                force_cleanup()
            print(f"  [{trace_tag}/{label}] done: ok={ok} oom={oom}")
            all_results[label] = records
    finally:
        trace_fp.close()

    # ── Summary table (full eval) ───────────────────────────────────────────
    print(f"\n{'='*100}\nFULL EVAL SUMMARY\n{'='*100}")
    print(f"{'Config':<48} {'VRAM':>7} {'TTFT':>7} {'ITL':>7} "
          f"{'EM%':>6} {'EM 95% CI':>16} {'F1%':>6} {'OOM':>4}")
    for label, recs in all_results.items():
        ok = [r for r in recs if r["status"] == "SUCCESS"]
        ooms = len(recs) - len(ok)
        if not ok:
            print(f"  {label:<48}  OOM"); continue
        max_vram = max(r["peak_vram_gb"] for r in ok)
        ttft = sum(r["ttft_sec"] for r in ok) / len(ok)
        itl  = sum(r["itl_sec"]  for r in ok) * 1000 / len(ok)
        em_vals = [1.0 if r["em_score"] else 0.0 for r in ok]
        f1_vals = [r["f1_tok"] for r in ok]
        em_m, em_lo, em_hi = bootstrap_ci(em_vals)
        f1_m, _, _         = bootstrap_ci(f1_vals)
        ci = f"[{em_lo*100:.1f},{em_hi*100:.1f}]"
        # [P3.3] Wilson CI cross-check
        n_succ = int(sum(em_vals))
        _, w_lo, w_hi = wilson_ci(n_succ, len(em_vals))
        print(f"  {label:<48} {max_vram:>5.2f}GB {ttft:>5.2f}s {itl:>5.0f}ms "
              f"{em_m*100:>5.1f}% {ci:>16} {f1_m*100:>5.1f}% {ooms:>4}  "
              f"Wilson95=[{w_lo*100:.1f},{w_hi*100:.1f}]")

    # ── [P3.2] Pager telemetry summary ─────────────────────────────────────
    print(f"\n{'='*100}\nPAGER TELEMETRY\n{'='*100}")
    print(f"{'Config':<48} {'Evict':>6} {'SLLM':>6} {'PgPeakVRAM':>12} {'AdaptSkip':>10}")
    for label, recs in all_results.items():
        ok = [r for r in recs if r["status"] == "SUCCESS"]
        if not ok:
            continue
        evicts = sum(r.get("eviction_events", 0) for r in ok)
        sllm   = sum(r.get("streamllm_offloads", 0) for r in ok)
        peak   = max(r["peak_vram_gb"] for r in ok)
        skipped = sum(1 for r in ok if r.get("adaptive_skipped"))
        print(f"  {label:<48} {evicts:>6} {sllm:>6} {peak:>10.2f}GB {skipped:>10}/{len(ok)}")

    # ── [P1.3] EM on the retrievable subset (gold-presence == True) ─────────
    if gold_hits:
        print(f"\n{'='*100}\nRETRIEVABLE-SUBSET EVAL (gold-presence == True)\n{'='*100}")
        retrievable = [q for q in queries if gold_hits.get(q, False)]
        n_retr = len(retrievable)
        print(f"  Subset size: {n_retr}/{len(queries)} questoes "
              f"(retrievability ceiling).")
        print(f"  {'Config':<48} {'EM%':>6} {'EM 95% CI':>16} {'F1%':>6}")
        for label, recs in all_results.items():
            ok = [r for r in recs if r["status"] == "SUCCESS"
                  and gold_hits.get(r["query"], False)]
            if not ok:
                continue
            em_vals = [1.0 if r["em_score"] else 0.0 for r in ok]
            f1_vals = [r["f1_tok"] for r in ok]
            em_m, em_lo, em_hi = bootstrap_ci(em_vals)
            f1_m, _, _         = bootstrap_ci(f1_vals)
            ci = f"[{em_lo*100:.1f},{em_hi*100:.1f}]"
            print(f"  {label:<48} {em_m*100:>5.1f}% {ci:>16} {f1_m*100:>5.1f}%")

    # ── [Fase 5.3] Paired bootstrap: contrastes pré-especificados ────────────
    # H0: μ(A) = μ(B). Reportar Δ, IC95 % e p-value bilateral. Bonferroni
    # com α/4 = 0.0125 (4 contrastes), critério: Δ>0 ∧ p<0.0125.
    print(f"\n{'='*100}\nPAIRED BOOTSTRAP (H0: Δ=0, two-sided)\n{'='*100}")
    print("  Bonferroni α=0.0125 (4 contrastes pré-especificados).")
    pairs_to_test = [
        # (label_a, label_b, scope_filter, description)
        ("C4: RAG + CRAG triage",
         "C1: Baseline No-RAG", "all",
         "C4 amplifica paramétrico (full eval)"),
        ("C4: RAG + CRAG triage",
         "C1: Baseline No-RAG", "retrievable",
         "C4 amplifica paramétrico (subset retrievable)"),
        ("C6: RAG + RAGAware Pager",
         "C4: RAG + CRAG triage", "retrievable",
         "C6 NÃO regrede vs C4 sem pager (canário)"),
        ("C8: Cognito-Adaptive (Gate+FS+CRAG+Pager)",
         "C1: Baseline No-RAG", "retrievable",
         "C8 preserva ganho no subset retrievable"),
    ]
    print(f"  {'Contraste':<58} {'metric':<4} {'ΔEM%':>7} {'IC95':>16} {'p':>7}")
    for la, lb, scope, desc in pairs_to_test:
        if la not in all_results or lb not in all_results:
            continue
        ra = {r["query"]: r for r in all_results[la]
              if r["status"] == "SUCCESS"}
        rb = {r["query"]: r for r in all_results[lb]
              if r["status"] == "SUCCESS"}
        common = [q for q in ra if q in rb]
        if scope == "retrievable" and gold_hits:
            common = [q for q in common if gold_hits.get(q, False)]
        if not common:
            continue
        em_a = [1.0 if ra[q]["em_score"] else 0.0 for q in common]
        em_b = [1.0 if rb[q]["em_score"] else 0.0 for q in common]
        f1_a = [ra[q]["f1_tok"] for q in common]
        f1_b = [rb[q]["f1_tok"] for q in common]
        d_em, lo_em, hi_em, p_em = paired_bootstrap_ci(em_a, em_b)
        d_f1, lo_f1, hi_f1, p_f1 = paired_bootstrap_ci(f1_a, f1_b)
        sig_em = "*" if p_em < 0.0125 else " "
        sig_f1 = "*" if p_f1 < 0.0125 else " "
        ci_em = f"[{lo_em*100:+.1f},{hi_em*100:+.1f}]"
        ci_f1 = f"[{lo_f1*100:+.1f},{hi_f1*100:+.1f}]"
        scope_tag = f"({scope},n={len(common)})"
        head = f"  {la[:18]:<18} − {lb[:18]:<18} {scope_tag:<18}"
        print(f"{head} EM   {d_em*100:>+6.1f} {ci_em:>16} {p_em:>5.3f}{sig_em} | {desc}")
        print(f"{'':<58} F1   {d_f1*100:>+6.1f} {ci_f1:>16} {p_f1:>5.3f}{sig_f1}")

    # ── [Fase 1.2] Pager-assert telemetry ───────────────────────────────────
    af = getattr(engine, "_pager_assert_failures", 0)
    pd = getattr(engine, "_pager_prefill_drift",   0)
    if af or pd:
        print(f"\n[Pager-Telemetry] piecewise≠mono: {af} | prefill_drift: {pd}")
        if af > 0:
            print("  ⚠ Tokenização piecewise diverge da monolítica → "
                  "register_segment_abs pode estar des-alinhado com input_ids reais.")
    else:
        print(f"\n[Pager-Telemetry] piecewise==mono em todas as queries; sem drift de prefill.")

    return all_results


def create_long_context(engine, base_passage: str, target_tokens: int = 4096) -> str:
    """[Fase 3.2] Default reduzido de 8192→4096. 8 k em Mistral-7B-NF4 + SDPA
    em T4 sem FA-2 satura attention map fp16 (peak ~6 GB) e OOMs antes do
    pager ativar. 4 k é suficiente para validar eviction com chunk_size=512."""
    enc = engine.tokenizer(base_passage, return_tensors="pt", add_special_tokens=False)
    sl  = enc["input_ids"].shape[-1]
    if sl == 0:
        return base_passage * 1000
    rep = (target_tokens // sl) + 1
    return ((base_passage + " ") * rep)[: target_tokens * 4]


# ─── Main ─────────────────────────────────────────────────────────────────
# ─── [P2A] NIAH (Needle-in-a-Haystack) — sintetico, RULER-style ─────────
# Hsieh et al. (2024) RULER. Versao single-key: 1 needle inserido em
# profundidade controlada num haystack de filler textual. Gold = literal do
# needle. Sem download HF; reproducivel via seed.
NIAH_FILLER_SENTENCES = [
    "The grass is green and the sky is blue.",
    "Water flows downhill and fire burns upward.",
    "Birds fly through clouds during the daytime.",
    "Mountains rise above valleys and rivers.",
    "Stars appear at night when the sun has set.",
    "Trees produce oxygen and consume carbon dioxide.",
    "Salt dissolves in water but oil does not.",
    "Sound travels slower than light in air.",
    "Iron rusts when exposed to oxygen and moisture.",
    "Glass shatters when struck with sufficient force.",
]

def _niah_filler(target_tokens: int, tokenizer) -> str:
    """Build filler text whose tokenization is approximately target_tokens.

    Uses len(tokenizer(text).input_ids) as ground truth; expands until close.
    """
    text = ""
    n_sent = 0
    while True:
        text = " ".join(NIAH_FILLER_SENTENCES * (n_sent // len(NIAH_FILLER_SENTENCES) + 1))[:9999999]
        toks = len(tokenizer(text, add_special_tokens=False).input_ids)
        if toks >= target_tokens:
            break
        n_sent += len(NIAH_FILLER_SENTENCES)
    # Trim to exactly target_tokens for determinism
    ids = tokenizer(text, add_special_tokens=False).input_ids[:target_tokens]
    return tokenizer.decode(ids)

def generate_niah_query(tokenizer, target_tokens: int, depth_pct: float,
                          seed: int = 0) -> dict:
    """Single-key NIAH (RULER-style).

    Returns dict with keys: query, gold, context, target_tokens, depth_pct.
    Needle: "The special access code for Cognito-{tag} is {value}."
    Query: "What is the special access code for Cognito-{tag}?"
    """
    import random
    rng = random.Random(seed)
    tag = f"{rng.randint(100, 999)}"
    value = f"{rng.randint(1000, 9999)}"
    needle = f"The special access code for Cognito-{tag} is {value}."
    query = f"What is the special access code for Cognito-{tag}?"
    gold = value

    # Build filler around needle so total ≈ target_tokens.
    needle_toks = len(tokenizer(needle, add_special_tokens=False).input_ids)
    filler_target = max(0, target_tokens - needle_toks)
    pre_target = int(filler_target * depth_pct)
    post_target = filler_target - pre_target

    pre = _niah_filler(pre_target, tokenizer) if pre_target > 32 else ""
    post = _niah_filler(post_target, tokenizer) if post_target > 32 else ""
    context = (pre + " " + needle + " " + post).strip()
    return {"query": query, "gold": gold, "context": context,
             "target_tokens": target_tokens, "depth_pct": depth_pct,
             "needle_tag": tag}

def build_niah_dataset(tokenizer, lengths=(4096, 8192),
                          depths=(0.10, 0.50, 0.90),
                          n_per_cell: int = 5) -> list:
    """Returns list of dicts: lengths × depths × n_per_cell items.
    Default = 2 × 3 × 5 = 30 queries (~30min em T4 com prompts 8k).
    """
    out = []
    seed = 0
    for L in lengths:
        for d in depths:
            for k in range(n_per_cell):
                seed += 1
                out.append(generate_niah_query(tokenizer, L, d, seed=seed))
    return out


# ─── H2O (Heavy-Hitter Oracle) Baseline — heuristic KV eviction ──────────
# Zhang et al. (NeurIPS 2023). Retains top-k tokens by accumulated attention
# + recency window. Contrast: Cognito uses retrieval-relevance scores.

class H2OEvictionPolicy:
    """H2O (Heavy-Hitter Oracle) — Zhang et al. NeurIPS 2023.

    Real implementation using L2-norm inverse of keys as attention proxy
    (Devoto et al. EMNLP 2024: low L2-norm keys attract more attention).
    Compatible with SDPA+NF4 without output_attentions=True.

    Keeps: sink_size attention sink tokens + heavy_ratio top-scoring
    heavy hitters + recent_ratio most-recent tokens. Evicts the rest.
    """

    def __init__(self, initial_threshold_gb: float = 5.5,
                 gpu_capacity_gb: float = 13.5,
                 heavy_ratio: float = 0.30, recent_ratio: float = 0.10,
                 sink_size: int = 4):
        self.threshold_bytes = initial_threshold_gb * (1024 ** 3)
        self.gpu_capacity_gb = gpu_capacity_gb
        self.heavy_ratio  = heavy_ratio
        self.recent_ratio = recent_ratio
        self.sink_size    = sink_size
        self.active       = False
        self._evictions   = 0
        self._streamllm_offloads = 0

    def reset(self):
        self._evictions = 0
        self._streamllm_offloads = 0

    @property
    def eviction_count(self) -> int:
        return self._evictions

    @property
    def streamllm_offloads(self) -> int:
        return self._streamllm_offloads

    @property
    def blocks_in_cpu(self) -> int:
        return 0

    def should_offload_predictive(self, cur_n: int = 1) -> bool:
        if not torch.cuda.is_available():
            return False
        # Use both current and recent-peak to catch post-prefill steady-state
        cur = torch.cuda.memory_allocated(0)
        return cur > self.threshold_bytes

    def _evict_layer(self, k: torch.Tensor, v: torch.Tensor,
                     heavy_budget: int, recent_budget: int):
        """H2O eviction on a single layer. Returns pruned (k, v)."""
        seq = k.shape[2]
        keep_total = self.sink_size + heavy_budget + recent_budget
        if keep_total >= seq:
            return k, v

        # L2-norm inverse: lower norm → higher attention (Devoto et al. 2024)
        importance = 1.0 / (torch.norm(k, p=2, dim=-1).clamp(min=1e-8))
        importance = importance.mean(dim=1)  # [batch, seq]

        batch = k.shape[0]
        nk_list, nv_list = [], []
        for b in range(batch):
            scores = importance[b]
            sink_idx   = torch.arange(self.sink_size, device=k.device)
            recent_idx = torch.arange(max(self.sink_size, seq - recent_budget),
                                       seq, device=k.device)
            # heavy hitters from non-sink, non-recent
            cand = torch.ones(seq, dtype=torch.bool, device=k.device)
            cand[:self.sink_size] = False
            cand[recent_idx] = False
            cand_scores = scores.clone()
            cand_scores[~cand] = -1.0
            n_heavy = min(heavy_budget, int(cand.sum()))
            _, heavy_idx = cand_scores.topk(k=n_heavy, dim=-1)

            keep, _ = torch.unique(
                torch.cat([sink_idx, heavy_idx, recent_idx])).sort()
            nk_list.append(k[b:b+1, :, keep, :])
            nv_list.append(v[b:b+1, :, keep, :])
        return torch.cat(nk_list, dim=0), torch.cat(nv_list, dim=0)

    def maybe_evict_pre_decode(self, past_kv, cur_len: int = None):
        """Apply H2O to all layers of past_key_values."""
        if past_kv is None or not self.should_offload_predictive():
            return past_kv
        try:
            key_cache, value_cache = _get_kv_tensors(past_kv)
        except Exception:
            return past_kv

        seq = key_cache[0].shape[2]
        heavy_budget  = max(self.sink_size, int(self.heavy_ratio  * seq))
        recent_budget = max(self.sink_size, int(self.recent_ratio * seq))

        new_k, new_v, evicted = [], [], 0
        for k, v in zip(key_cache, value_cache):
            nk, nv = self._evict_layer(k, v, heavy_budget, recent_budget)
            new_k.append(nk); new_v.append(nv)
            evicted = max(evicted, k.shape[2] - nk.shape[2])

        if evicted > 0:
            self._evictions += 1
            if torch.cuda.is_available():
                torch.cuda.synchronize(); torch.cuda.empty_cache()
            if self._evictions <= 3:
                print(f"[H2O] Evict #{self._evictions}: {seq}→{new_k[0].shape[2]} tok "
                      f"(kept {heavy_budget} heavy + {recent_budget} recent "
                      f"+ {self.sink_size} sinks)")
        return _rebuild_cache_from_tensors(new_k, new_v)

    def offload_kv_cache(self, past_kv):
        return past_kv

# ─── [P2B] Long-context QA loader (públicos, sem auth) ───────────────────
def load_narrativeqa(tokenizer, n_per_split: int = 15) -> tuple:
    """Load long-context QA from public datasets (no HF_TOKEN required).

    Priority:
    1. allenai/qasper  — scientific QA, ~5k tokens/doc (Dasigi et al. 2021)
    2. rajpurkar/squad  — SQuAD passages expanded with NIAH-style filler
                          to reach ≥ 4k tokens (simulates long-context pressure)
    Returns: (queries, gold_answers, pre_fetched_contexts, pre_fetched_scores)
    """
    from datasets import load_dataset

    queries, gold, contexts, scores = [], {}, {}, {}

    # ─── Attempt 1: Qasper (allenai/qasper) ──────────────────────────
    try:
        print("[LongCtxQA] Trying allenai/qasper (public)...")
        ds = load_dataset("allenai/qasper", split="test", trust_remote_code=True)
        for paper in ds:
            if len(queries) >= n_per_split:
                break
            # Build full paper text from sections
            full_text_parts = []
            for section, paras in zip(
                paper.get("full_text", {}).get("section_name", []),
                paper.get("full_text", {}).get("paragraphs", [])
            ):
                full_text_parts.append("## " + str(section) + "\n" + " ".join(paras))
            ctx = paper.get("abstract", "") + "\n\n" + "\n\n".join(full_text_parts)

            # Pick first answerable QA pair
            for qa in paper.get("qas", []):
                q = qa.get("question", "").strip()
                if not q:
                    continue
                ans_list = []
                for ann in qa.get("answers", []):
                    ans = ann.get("answer", {})
                    if ans.get("unanswerable"):
                        continue
                    spans = ans.get("extractive_spans") or []
                    free  = ans.get("free_form_answer", "")
                    if spans:
                        ans_list.extend(spans)
                    elif free:
                        ans_list.append(free)
                if not ans_list or not ctx:
                    continue
                queries.append(q)
                gold[q] = ans_list[:3]   # up to 3 aliases
                contexts[q] = ctx[:120_000]  # cap at ~120k chars (~30k tok)
                scores[q] = [(contexts[q], 1.0)]
                break  # one QA per paper
        if queries:
            avg_tok = sum(
                len(tokenizer(contexts[q], add_special_tokens=False).input_ids)
                for q in queries
            ) // len(queries)
            print(f"[LongCtxQA] ✅ Loaded {len(queries)} from qasper "
                  f"(avg ~{avg_tok} tok/doc)")
            return queries, gold, contexts, scores
    except Exception as e1:
        print(f"[LongCtxQA] qasper failed ({type(e1).__name__}: {e1})")

    # ─── Attempt 2: SQuAD + NIAH-style filler to ~4k tokens ──────────
    try:
        print("[LongCtxQA] Trying rajpurkar/squad + 4k filler...")
        ds = load_dataset("rajpurkar/squad", split="validation",
                          trust_remote_code=True)
        seen_q = set()
        for item in ds:
            if len(queries) >= n_per_split:
                break
            q = item["question"].strip()
            if q in seen_q:
                continue
            ans_text = item["answers"]["text"]
            if not ans_text:
                continue
            base_ctx = item["context"]
            # Expand to ~4k tokens with NIAH filler so pager is pressured
            filler = " ".join(NIAH_FILLER_SENTENCES * 80)  # ~4k tok
            ctx = filler[:len(filler)//2] + " " + base_ctx + " " + filler[len(filler)//2:]
            seen_q.add(q)
            queries.append(q)
            gold[q] = ans_text[:3]
            contexts[q] = ctx
            scores[q] = [(ctx, 1.0)]
        if queries:
            avg_tok = sum(
                len(tokenizer(contexts[q], add_special_tokens=False).input_ids)
                for q in queries
            ) // len(queries)
            print(f"[LongCtxQA] ✅ Loaded {len(queries)} from squad+filler "
                  f"(avg ~{avg_tok} tok/doc)")
            return queries, gold, contexts, scores
    except Exception as e2:
        print(f"[LongCtxQA] squad failed ({type(e2).__name__}: {e2})")

    print("[LongCtxQA] All public sources failed; aborting.")
    return [], {}, {}, {}

# ─── [P2 Stress] NIAH at 16k/32k depths (deep pressure test) ──────────────
def build_niah_stress_dataset(tokenizer, lengths=(16384, 32768),
                                 depths=(0.25, 0.50, 0.75),
                                 n_per_cell: int = 3) -> list:
    """Stress variant: deeper contexts, fewer samples for time budget.
    Default = 2 × 3 × 3 = 18 queries.
    """
    return build_niah_dataset(tokenizer, lengths=lengths, depths=depths, n_per_cell=n_per_cell)


# ─── [LitM] Lost-in-the-Middle Mitigations ────────────────────────────────
# Liu et al. (2024) "Lost in the Middle": modelos ignoram informação em
# posições intermediárias. Três camadas de mitigação implementadas em userland.

def _split_context_chunks(context: str, tokenizer, chunk_size: int = 256) -> list:
    """Split context into ~chunk_size token chunks for BM25 scoring."""
    tokens = tokenizer(context, add_special_tokens=False).input_ids
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk_ids = tokens[i:i + chunk_size]
        chunks.append(tokenizer.decode(chunk_ids))
    return [c for c in chunks if c.strip()]

def trim_context_bm25(context: str, query: str, tokenizer,
                       max_tokens: int = 8192, chunk_size: int = 256) -> str:
    """[Camada 3 LitM] BM25-based context trimming (quasi-LongLLMLingua).

    Splits context into chunks, scores each vs query via BM25, keeps top-K
    chunks up to max_tokens. Reduces 16k/32k → max_tokens without a
    compression model.
    """
    # Guard: se contexto já cabe em max_tokens, só reordena sem trimar
    ctx_toks = len(tokenizer(context, add_special_tokens=False).input_ids)
    if ctx_toks <= max_tokens:
        return reorder_context_lim(context, query, tokenizer, chunk_size)

    chunks = _split_context_chunks(context, tokenizer, chunk_size)
    if not chunks:
        return context

    # BM25 score each chunk against query
    tokenized = [re.findall(r"\w+", c.lower()) for c in chunks]
    bm25 = BM25Okapi(tokenized)
    q_tokens = re.findall(r"\w+", query.lower())
    scores = bm25.get_scores(q_tokens)

    # Sort chunks by score (descending), keep until max_tokens
    ranked = sorted(zip(scores, range(len(chunks))), reverse=True)
    kept_indices = set()
    total_toks = 0
    for score, idx in ranked:
        chunk_toks = len(tokenizer(chunks[idx], add_special_tokens=False).input_ids)
        if total_toks + chunk_toks > max_tokens:
            break
        kept_indices.add(idx)
        total_toks += chunk_toks

    # Reorder kept chunks with LIM (most relevant at start and end)
    kept = sorted(kept_indices)  # preserve original order first
    kept_scored = sorted([(scores[i], i) for i in kept], reverse=True)

    # Apply Lost-in-Middle reordering to kept chunks
    n = len(kept_scored)
    if n <= 2:
        reordered = [chunks[i] for _, i in kept_scored]
    else:
        out = [None] * n
        left, right = 0, n - 1
        for j, (_, idx) in enumerate(kept_scored):
            if j % 2 == 0:
                out[left] = chunks[idx]; left += 1
            else:
                out[right] = chunks[idx]; right -= 1
        reordered = out

    return " ".join(reordered)

def reorder_context_lim(context: str, query: str, tokenizer,
                         chunk_size: int = 512) -> str:
    """[Camada 2 LitM] Lost-in-Middle reordering without trimming.

    Splits context into chunks, scores by BM25, reorders so most relevant
    chunks appear at start and end (Liu et al. 2024 recommendation).
    Top-1 chunk (highest BM25 score) always goes first — most important
    position for causal LMs (primacy effect, Zhao et al. 2021).
    """
    chunks = _split_context_chunks(context, tokenizer, chunk_size)
    if len(chunks) <= 2:
        return context
    tokenized = [re.findall(r"\w+", c.lower()) for c in chunks]
    bm25 = BM25Okapi(tokenized)
    q_tokens = re.findall(r"\w+", query.lower())
    scores = bm25.get_scores(q_tokens)
    ranked = sorted(enumerate(scores), key=lambda x: -x[1])

    # Top-1 always first (primacy), rest via LIM alternating start/end
    top_idx = ranked[0][0]
    rest = [idx for idx, _ in ranked[1:]]
    n_rest = len(rest)
    out = [chunks[top_idx]]
    if n_rest > 0:
        lim_out = [None] * n_rest
        left, right = 0, n_rest - 1
        for j, idx in enumerate(rest):
            if j % 2 == 0:
                lim_out[left] = chunks[idx]; left += 1
            else:
                lim_out[right] = chunks[idx]; right -= 1
        out.extend(lim_out)
    return " ".join(out)

def main():
    is_test = "--test" in sys.argv
    is_longctx = "--longctx" in sys.argv
    is_narrativeqa = "--narrativeqa" in sys.argv
    is_stress_deep = "--stress-deep" in sys.argv
    is_clear_logs = "--clear-logs" in sys.argv
    is_clear_traces = "--clear-traces" in sys.argv
    if is_clear_traces:
        open(EVAL_TRACE_PATH, "w").close()
        print(f"[Traces] Cleared {EVAL_TRACE_PATH}")
    if not os.path.exists("gold_answers.json"):
        print("AVISO: gold_answers.json nao encontrado. Execute a ingestao (Cell 5) primeiro.")
        return
    with open("gold_answers.json", "r") as f:
        gold_answers = json.load(f)

    # [P2A] Engine init must happen before the longctx branch (which uses
    # engine.tokenizer to build the NIAH dataset).
    pager = RAGAwarePager(initial_threshold_gb=7.5, page_size=1024,
                            safety_margin=0.15, ema_alpha=0.3,
                            warmup_steps=10, gpu_capacity_gb=15.5)
    engine = HardenedLLMEngine(model_name=LLM_MODEL_NAME,
                                  vector_db_path=VECTOR_DB_PATH,
                                  paging_manager=pager)
    engine.initialize_runtime()

    # ─── [P2A] Long-context branch ─────────────────────────────────────────
    if is_longctx:
        # [Fix] Restore T4-realistic memory cap (init defaults to 35% for
        # stress tests; longctx needs full T4 to make L1 a fair baseline).
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.85, device=0)
            torch.cuda.empty_cache()
            print("[LongCtx] Memory cap raised to 85% (~13.5GB on T4-16GB).")
        # ─── [P2] Eval logging management ────────────────────────────────
        if is_clear_logs:
            open(EVAL_LOG_PATH, "w").close()
            print(f"[Logs] Cleaned {EVAL_LOG_PATH}")
            append_log(f"\n{'='*88}")
            append_log(f"[EVAL SESSION] Started {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            eval_type = "P2B NarrativeQA" if is_narrativeqa else "P2 Stress 16k/32k" if is_stress_deep else "P2 NIAH 4k/8k"
            append_log(f"[EVAL TYPE] {eval_type}")
            append_log(f"{'='*88}")

    # ─── [P2B/P2-Stress] Branch selector ─────────────────────────────
        if is_narrativeqa:
            print("\n[NarrativeQA] Loading LongBench single-doc QA…")
            lc_queries, lc_gold, lc_prefetch, lc_scores = load_narrativeqa(engine.tokenizer,
                                                                              n_per_split=(5 if is_test else 15))
            if not lc_queries:
                print("[NarrativeQA] Load failed; aborting.")
                return
        elif is_stress_deep:
            print("\n[Stress-Deep] Building NIAH 16k/32k dataset…")
            if is_test:
                niah_data = build_niah_stress_dataset(engine.tokenizer,
                                                        lengths=(8192,), depths=(0.5,),
                                                        n_per_cell=2)
            else:
                niah_data = build_niah_stress_dataset(engine.tokenizer,
                                                        lengths=(16384, 32768),
                                                        depths=(0.25, 0.50, 0.75),
                                                        n_per_cell=3)
            lc_queries = [d["query"] for d in niah_data]
            lc_gold = {d["query"]: [d["gold"]] for d in niah_data}
            lc_prefetch = {d["query"]: d["context"] for d in niah_data}
            lc_scores = {d["query"]: [(d["context"], 1.0)] for d in niah_data}
        else:
            # P2 default: NIAH 4k/8k
            print("\n[LongCtx] Building NIAH dataset (sintetico)…")
            if is_test:
                niah_data = build_niah_dataset(engine.tokenizer,
                                                lengths=(2048,), depths=(0.5,),
                                                n_per_cell=2)
            else:
                niah_data = build_niah_dataset(engine.tokenizer,
                                                lengths=(4096, 8192),
                                                depths=(0.10, 0.50, 0.90),
                                                n_per_cell=5)
            lc_queries = [d["query"] for d in niah_data]
            lc_gold = {d["query"]: [d["gold"]] for d in niah_data}
            lc_prefetch = {d["query"]: d["context"] for d in niah_data}
            lc_scores = {d["query"]: [(d["context"], 1.0)] for d in niah_data}

        print(f"[LongCtx] {len(lc_queries)} queries loaded.")

        # [Fix L2/L3] Calibration aligned with T4-16GB headroom.
        # Old logic: probe at 256 tok with use_cache=False → ~weights only.
        # thr = peak_probe*0.85 fired eviction at any prompt > probe size,
        # dropping the entire ctx_coarse (and the needle). New: threshold
        # near OOM boundary so eviction is a genuine pressure-relief valve.
        # T4-16GB usable ~13.5 GB; reserve 1.5 GB safety → 12.0 GB threshold.
        EVICT_THRESHOLD_GB = 5.5   # dispara em queries >=8k (VRAM ~5.83 GB)
        lc_pager = RAGAwarePager(initial_threshold_gb=EVICT_THRESHOLD_GB,
                                   gpu_capacity_gb=13.5)
        engine.pager = lc_pager
        print(f"[LongCtx] eviction_threshold={EVICT_THRESHOLD_GB:.2f} GB "
              f"(dispara em queries >=8k).")

        # BM25-scored chunks para L3: divide cada contexto em chunks e
        # dá score BM25 vs query — RAGAwarePager preserva o chunk mais relevante.
        def _bm25_chunks(context, query, tokenizer, chunk_size=256, max_chunks=12):
            toks = tokenizer(context, add_special_tokens=False).input_ids
            chunks = [tokenizer.decode(toks[i:i+chunk_size])
                      for i in range(0, len(toks), chunk_size)][:max_chunks]
            if not chunks:
                return [(context, 1.0)]
            tokenized = [re.findall(r"\w+", c.lower()) for c in chunks]
            bm25 = BM25Okapi(tokenized)
            q_toks = re.findall(r"\w+", query.lower())
            raw = bm25.get_scores(q_toks)
            mn, mx = raw.min(), raw.max()
            normed = (raw - mn) / (mx - mn) if mx > mn else [1.0]*len(chunks)
            return [(c, float(s)) for c, s in zip(chunks, normed)]

        lc_bm25_scores = {
            q: sorted(_bm25_chunks(lc_prefetch[q], q, engine.tokenizer),
                       key=lambda x: -x[1])   # sorted by BM25 score desc
            for q in lc_queries}
        n_chunks = sum(len(v) for v in lc_bm25_scores.values())
        print(f"[LongCtx] BM25 segments: {n_chunks} chunks / {len(lc_queries)} queries "
              f"(avg {n_chunks//len(lc_queries)} chunks/query)")

        LONGCTX_CFGS = [
            {"label": "L1: NoPaging",          "use_rag": True,
             "use_paging": False},
            {"label": "L2: ChunkedPrefill",     "use_rag": True,
             "use_paging": False, "use_chunked_prefill": True},
            {"label": "L3: Cognito (Chunked+SegmentAware)", "use_rag": True,
             "use_paging": True,  "use_chunked_prefill": True,
             "use_passage_prefill": False},  # Path B: coarse segment, threshold-driven
            {"label": "L4: H2O (Attention-based)",  "use_rag": True,
             "use_paging": True,  "use_chunked_prefill": True,
             "h2o_pager": True},
            # L5: Query Repetition only (Reorder/Trim fragment NIAH needle across chunks).
            {"label": "L5: Cognito+QRepeat (LitM)", "use_rag": True,
             "use_paging": True,  "use_chunked_prefill": True,
             "use_passage_prefill": False,
             "use_query_repetition": True,
             "use_context_reorder": False,
             "use_context_trim": False},
        ]

        # Run L1-L3 with RAGAwarePager; L4 with H2O; L5 with RAGAwarePager + LitM.
        rag_pager = engine.pager
        l5_cfgs = [LONGCTX_CFGS.pop()]  # L5 QRepeat
        l4_cfg   = LONGCTX_CFGS.pop()   # L4 H2O
        l3_cfg   = LONGCTX_CFGS.pop()   # L3 Cognito (BM25 segments)
        # L1/L2: score uniforme (sem discriminação de segmentos)
        run_evaluation(lc_queries, LONGCTX_CFGS, engine, lc_gold,
                        max_new_tokens=32,
                        pre_fetched=lc_prefetch,
                        pre_fetched_scores=lc_scores,
                        gold_hits=None,
                        verbose=is_test, trace_tag="longctx")
        # L3: BM25 scores reais — RAGAwarePager preserva chunk mais relevante
        run_evaluation(lc_queries, [l3_cfg], engine, lc_gold,
                        max_new_tokens=32,
                        pre_fetched=lc_prefetch,
                        pre_fetched_scores=lc_bm25_scores,
                        gold_hits=None,
                        verbose=is_test, trace_tag="longctx")

        # Run L4 with H2O pager
        # threshold=5.0 GB: KV steady-state para 8k ≈ 5.5 GB > 5.0 → evicta
        # Para 4k: steady-state ≈ 4.5 GB < 5.0 → não evicta (controle limpo)
        h2o_pager = H2OEvictionPolicy(initial_threshold_gb=5.0,
                                      gpu_capacity_gb=13.5,
                                      heavy_ratio=0.30, recent_ratio=0.10,
                                      sink_size=4)
        engine.pager = h2o_pager
        run_evaluation(lc_queries, [l4_cfg], engine, lc_gold,
                        max_new_tokens=32,
                        pre_fetched=lc_prefetch,
                        pre_fetched_scores=lc_scores,
                        gold_hits=None,
                        verbose=is_test, trace_tag="longctx")

        # Run L5 (QRepeat) with RAGAwarePager + Query Repetition
        engine.pager = rag_pager
        run_evaluation(lc_queries, l5_cfgs, engine, lc_gold,
                        max_new_tokens=32,
                        pre_fetched=lc_prefetch,
                        pre_fetched_scores=lc_scores,
                        gold_hits=None,
                        verbose=is_test, trace_tag="longctx")
        engine.pager = rag_pager
        append_log(f"[EVAL COMPLETE] All configs finished at {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        append_log("See notebook output above for detailed telemetry.\n")
        return  # skip factoid main + 4k stress

    eval_queries = list(gold_answers.keys())[:(2 if is_test else NUM_QUERIES_EVAL)]

    # [P1.3] Load gold-presence map (from corpus ingestion).
    gold_hits = None
    if os.path.exists("corpus_gold_hits.json"):
        with open("corpus_gold_hits.json", "r") as f:
            gold_hits = json.load(f)
        n_hits = sum(1 for q in eval_queries if gold_hits.get(q, False))
        print(f"[GoldHits] Retrievable subset: {n_hits}/{len(eval_queries)} eval queries.")
    else:
        print("[GoldHits] corpus_gold_hits.json missing — re-run ingestion to enable retrievable-subset eval.")

    # Seed FewShotPool from disjoint tail of gold set (no eval contamination).
    seed_n = min(5, max(0, len(gold_answers) - len(eval_queries)))
    if seed_n > 0:
        seed_qs = list(gold_answers.keys())[-seed_n:]
        for q in seed_qs:
            answers = gold_answers.get(q) or []
            if answers:
                engine.fewshot.add(q, answers[0])
        print(f"[FewShot] Seeded {seed_n} ICL exemplars.")

    CONFIGURATIONS = [
        {"label": "C1: Baseline No-RAG",                   "use_rag": False, "use_paging": False},
        {"label": "C2: RAG + FastPath (PromptLookup)",     "use_rag": True,  "use_paging": False},
        {"label": "C3: RAG + Few-shot ICL",                "use_rag": True,  "use_paging": False, "use_fewshot": True},
        {"label": "C4: RAG + CRAG triage",                 "use_rag": True,  "use_paging": False, "use_crag": True},
        {"label": "C5: RAG + Few-shot + CRAG",             "use_rag": True,  "use_paging": False, "use_fewshot": True, "use_crag": True},
        {"label": "C6: RAG + RAGAware Pager",              "use_rag": True,  "use_paging": True,  "use_passage_prefill": True},
        {"label": "C7: Cognito-Full (FS+CRAG+Pager)",      "use_rag": True,  "use_paging": True,  "use_passage_prefill": True, "use_fewshot": True, "use_crag": True},
        # [P1.1] C8: Cognito-Adaptive — gate adaptativo (Mallen 2023, Self-RAG 2024)
        {"label": "C8: Cognito-Adaptive (Gate+FS+CRAG+Pager)", "use_rag": True, "use_paging": True,
         "use_passage_prefill": True, "use_fewshot": True, "use_crag": True,
         "use_adaptive_gate": True},
    ]

    # ── [Fase 2.1] Calibração empírica do gate adaptativo ──────────────────
    # Pre-fetch retrieval ANTES de rodar eval, para calibrar o threshold do
    # gate em função do histograma observado de top-1 cross-encoder scores.
    # Sem isso, C8 fica preso em RAG_off (>96 % das queries) — não testável.
    print("\n[Eval] Pre-fetching retrieval para calibração de gate e eval principal...")
    pre_fetched_full, pre_scores_full = {}, {}
    for q in eval_queries:
        ds = engine.retrieve_with_scores(q)
        pre_scores_full[q] = ds
        pre_fetched_full[q] = "\n".join([d for d, _ in ds])

    thr_cal, fire_cal, n_cal = calibrate_adaptive_gate(
        pre_scores_full, target_fire_rate=ADAPTIVE_GATE_TARGET_FIRE)
    lo_b, hi_b = ADAPTIVE_GATE_FIRE_BOUNDS
    print(f"[Fase 2.1] Gate calibrado: threshold={thr_cal:.3f} → "
          f"fire_rate={fire_cal*100:.0f}% sobre n={n_cal} queries.")
    if not (lo_b <= fire_cal <= hi_b):
        print(f"  ⚠ fire_rate fora de [{lo_b*100:.0f}%, {hi_b*100:.0f}%]; "
              f"distribuição de top1 muito concentrada — considerar gate por dispersão.")
    # Rebind do módulo-level threshold usado por should_retrieve_adaptive.
    global ADAPTIVE_RETRIEVAL_GATE
    ADAPTIVE_RETRIEVAL_GATE = float(thr_cal)

    # Pre-extract stress context BEFORE eval (which deletes embed/rerank models).
    # [Fase 3.2] target_tokens reduzido 8k→4k (ver create_long_context).
    stress_queries = eval_queries[:10]
    ds_stress = engine.retrieve_with_scores(stress_queries[0]) if stress_queries else []
    long_ctx  = create_long_context(engine, ds_stress[0][0] if ds_stress else "Test context",
                                      target_tokens=4096)
    pre_f_stress = {q: long_ctx for q in stress_queries}
    pre_s_stress = {q: [(long_ctx, 0.5)] for q in stress_queries}

    run_evaluation(eval_queries, CONFIGURATIONS, engine, gold_answers,
                    pre_fetched=pre_fetched_full,
                    pre_fetched_scores=pre_scores_full,
                    gold_hits=gold_hits,
                    verbose=is_test, trace_tag="main")

    # ── [Fase 3] Stress test redesign ────────────────────────────────────────
    # 3.1: cap subido de 25%/35%→85% (~13.6 GB em T4-16). O cap baixo
    #      estrangulava o test, não o pager — peak observado=4.38 GB com
    #      cap=5.4 GB causava OOM antes do pager ativar.
    # 3.2: prompt 8k→4k.
    # 3.3: auto-calibrar eviction_threshold com base no peak natural.
    print("\n\n>>> STRESS TESTS (VRAM 85%, 4k tokens) <<<")
    if torch.cuda.is_available():
        torch.cuda.set_per_process_memory_fraction(0.85, device=0)
        torch.cuda.empty_cache()
        # Peak natural antes do pager: rodar uma query S1-equivalente como sonda.
        try:
            probe_in = engine.tokenizer(long_ctx + "\nQuestion: probe",
                                         return_tensors="pt",
                                         add_special_tokens=False).to(
                                             next(engine.model.parameters()).device)
            torch.cuda.reset_peak_memory_stats()
            with torch.no_grad():
                _ = engine.model(input_ids=probe_in["input_ids"][:, :128], use_cache=False)
            peak_probe = torch.cuda.max_memory_allocated() / (1024 ** 3)
        except Exception as e:
            peak_probe = 5.0
            print(f"[Fase 3.3] Probe falhou ({type(e).__name__}); fallback peak=5.0 GB")
    else:
        peak_probe = 5.0
    # Threshold = 0.85 × peak_probe (auto-calibração; Fase 3.3).
    stress_thr = max(2.0, peak_probe * 0.85)
    stress_pager = PredictiveMemoryPolicy(initial_threshold_gb=stress_thr,
                                            gpu_capacity_gb=13.5)
    stress_pager.auto_calibrate(peak_probe, headroom_gb=0.0)
    print(f"[Fase 3.3] peak_probe={peak_probe:.2f} GB → eviction_threshold="
          f"{stress_thr:.2f} GB.")
    engine.pager = stress_pager

    STRESS_CFGS = [
        {"label": "S1: NoPaging (4k)",                "use_paging": False},
        {"label": "S2: Predictive Paging (4k)",       "use_paging": True},
        {"label": "S3: ChunkedPrefill+Paging (4k)",   "use_paging": True, "use_chunked_prefill": True},
    ]
    run_evaluation(stress_queries, STRESS_CFGS, engine, gold_answers,
                    pre_fetched=pre_f_stress, pre_fetched_scores=pre_s_stress,
                    gold_hits=gold_hits,
                    verbose=is_test, trace_tag="stress")


if __name__ == "__main__":
    main()


Overwriting 3_inferencia.py


In [13]:
!uv run 3_inferencia.py --longctx --clear-traces --clear-logs
!uv run 3_inferencia.py --longctx --narrativeqa
!uv run 3_inferencia.py --longctx --stress-deep

[Traces] Cleared eval_traces.jsonl
[Engine] PyTorch alloc capped at 35% (~5.4GB on T4-16GB).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [01:06<00:00,  4.38it/s]
[Engine] attn_implementation=sdpa active.
[RAGAware] Threshold: 3.86+0.90=4.76 GB
<All keys matched successfully>
Loading weights: 100% 105/105 [00:00<00:00, 861.30it/s]
[LongCtx] Memory cap raised to 85% (~13.5GB on T4-16GB).
[Logs] Cleaned eval_results.txt

[LongCtx] Building NIAH dataset (sintetico)…
[LongCtx] 30 queries loaded.
[LongCtx] eviction_threshold=5.50 GB (dispara em queries >=8k).
[LongCtx] BM25 segments: 360 chunks / 30 queries (avg 12 chunks/query)
[Retrieval@3]  recall=100.0% MRR=1.000 (n=30)
[Retrieval@10] recall=100.0% MRR=1.000 (n=30)

[longctx] === L1: NoPaging ===
  [  1/30] SUCC EM=True F1=1.00 vram=10.02GB ttft=7.68s itl=467ms
  [ 10/30] SUCC EM=True F1=1.00 vram=10.02GB ttft=8.30s itl=505ms
[Engine] fast-path failed (OutOfMemoryError); vanilla fallback

In [6]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE FIM DE SESSÃO — Execute ANTES de fechar o Colab
# ═══════════════════════════════════════════════════════════════
# Persiste dados locais de volta ao Drive.
# IMPORTANTE: rodar antes de encerrar a sessão ou antes do timeout.

import os, shutil, time, json as _j

DRIVE_ROOT = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT = '/content'

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

saved = []

# ── Salva ChromaDB ────────────────────────────────────────────────
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
if os.path.exists(f'{LOCAL_CHROMA}/chroma.sqlite3'):
    print('Salvando ChromaDB → Drive ...')
    t0 = time.time()
    if os.path.exists(DRIVE_CHROMA):
        shutil.rmtree(DRIVE_CHROMA)
    shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DRIVE_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB salvo: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
    saved.append('chroma_cognito')
else:
    print('  ⚠ ChromaDB não encontrado em /content/ — nada a salvar.')

# ── Salva gold_answers.json ───────────────────────────────────────
LOCAL_GOLD = f'{LOCAL_ROOT}/gold_answers.json'
if os.path.exists(LOCAL_GOLD):
    shutil.copy(LOCAL_GOLD, f'{DRIVE_ROOT}/gold_answers.json')
    print(f'  ✓ gold_answers.json salvo.')
    saved.append('gold_answers.json')

# ── Salva checkpoints JSONL ───────────────────────────────────────
ckpt_files = [f for f in os.listdir(LOCAL_ROOT)
              if f.startswith('results_checkpoint_') and f.endswith('.jsonl')]
for ck in ckpt_files:
    src = f'{LOCAL_ROOT}/{ck}'
    dst = f'{DRIVE_ROOT}/checkpoints/{ck}'
    shutil.copy(src, dst)
    # Conta linhas (queries completadas)
    n_lines = sum(1 for _ in open(src))
    print(f'  ✓ {ck}: {n_lines} queries salvas.')
    saved.append(ck)

# ── Relatório ─────────────────────────────────────────────────────
print()
if saved:
    print(f'✓ Sessão salva no Drive: {saved}')
    print(f'  Caminho: {DRIVE_ROOT}')
else:
    print('⚠ Nada foi salvo — verifique os caminhos.')

# ── Status dos checkpoints (progresso da ablação) ─────────────────
print()
print('Status da Ablação:')
all_ckpts = [f for f in os.listdir(f'{DRIVE_ROOT}/checkpoints') if f.endswith('.jsonl')]
if all_ckpts:
    for ck in sorted(all_ckpts):
        path = f'{DRIVE_ROOT}/checkpoints/{ck}'
        lines = open(path).readlines()
        ooms = sum(1 for l in lines if '"OOM_FAILURE"' in l)
        ok = len(lines) - ooms
        print(f'  {ck:<45} {ok:>3} OK  {ooms:>3} OOM  ({len(lines):>3} total)')
else:
    print('  Nenhum checkpoint encontrado ainda.')

Salvando ChromaDB → Drive ...
  ✓ ChromaDB salvo: 92.2 MB em 1.1s
  ✓ gold_answers.json salvo.

✓ Sessão salva no Drive: ['chroma_cognito', 'gold_answers.json']
  Caminho: /content/drive/MyDrive/cognito_tcc

Status da Ablação:
  Nenhum checkpoint encontrado ainda.
